# Vectorless RAG Pipeline (PageIndex) — FinanceBench (full batch run)

**Paper:** Lumer et al. (2025), arXiv 2511.18177
**Library:** `pageindex` (PyPI, pinned `0.2.12`) — installed via pip, not the
vendored `external/PageIndex` git clone the previous version of this notebook
used. That folder is gitignored ("cloned, not vendored"), so it never existed
on a fresh Colab clone; the pip package covers the same functions
(`page_index_main`, `ConfigLoader`, `get_page_tokens`, `create_node_mapping`,
`llm_completion`) at the top level. See `vectorless_rag_walkthrough.ipynb` for
the debugging trail that found this, plus the asyncio/rate-limit issues below.

## What this pipeline is

No embeddings, no chunking. Each document is parsed once into a hierarchical
tree index (a table-of-contents structure with per-section titles and short
LLM-written summaries), and an LLM navigates that tree to pick the section(s)
most likely to hold the answer — no vector similarity search involved. Only
the raw text of the *selected* sections is passed to the generator. Failure
mode per CLAUDE.md: incorrect navigation to the wrong section.

## Scope decisions carried over from the walkthrough

- **Model: `gemini-3.1-flash-lite`, not `gemini-3.5-flash`.** Free tier caps
  `gemini-3.5-flash` at 5 requests/minute, and tree-building fires many
  concurrent calls — hit that wall directly in the walkthrough. flash-lite has
  a much bigger free-tier allowance. This also matches `vector_rag_pipeline.ipynb`,
  which switched for the same reason — keeping the same generation model across
  pipelines matters for a fair accuracy-latency-cost comparison.
- **`nest_asyncio.apply()` is required.** `page_index_main` calls its own
  `asyncio.run()` internally; a notebook kernel already has one running, so
  without this patch tree-building raises
  `RuntimeError: asyncio.run() cannot be called from a running event loop`.
- **Parser: PyPDF2 (PageIndex's default)**, not pymupdf4llm. Lower text
  quality on tables, but no extra dependency, and it's the parser
  `page_index_main` already uses internally to decide section page ranges —
  matching it here means no risk of page boundaries desyncing.
- **Resumable.** Every document/question already saved to disk is skipped on
  rerun. A real 84-document, 150-question run realistically won't finish in
  one sitting, and free-tier rate limits mean some calls will fail outright.
- **`gemini-3.1-flash-lite` billed at $0.25/1M input, $1.50/1M output**
  (confirmed against ai.google.dev/gemini-api/docs/pricing) — real money if
  billing is attached to the API key, since Google may bill directly instead
  of falling back to the free tier once a payment method is on the account.
  Use a billing-free key to actually get the free tier.

## Pipeline stages

```
Stage 0  Setup            — Drive mount, repo clone/pull, API keys, output paths
Stage 1  Load data        — 150 questions, 84 unique documents
Stage 2  Build trees      — PDF -> hierarchical section tree, one per document (resumable)
Stage 3  Navigate         — LLM picks node_id(s) likely to hold the answer (resumable)
Stage 4  Generate         — raw text of selected node(s) + question -> answer (resumable)
Stage 5  Preview          — sample of generated answers next to gold answers
Stage 6  Score            — deterministic match + LLM judge fallback (resumable)
Stage 8  Latency          — dedicated sequential timing pass, median-of-N (resumable, costs extra)
Stage 9  Summarize        — answer quality + navigation hit rate + token/cost totals
```

Stages 6-9 reuse `evaluation/answer_scorer.py` and `evaluation/cost_tracker.py` — the same shared modules `vector_rag_pipeline.ipynb`
uses, so both pipelines are scored, measured, and costed identically.

---
## Stage 0 — Setup

In [1]:
import os, sys, json, re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google.colab import drive

drive.mount('/content/drive')

REPO_ROOT = Path('/content/drive/MyDrive/financebench_project')
if not (REPO_ROOT / "data" / "financebench_open_source.jsonl").exists():
    print(f"Repo not found at {REPO_ROOT} — cloning ...")
    !git clone https://github.com/shaliqsv/financebench-rag-thesis.git "{REPO_ROOT}"
else:
    print(f"Repo already present at {REPO_ROOT} — syncing to latest main ...")
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" checkout main
    !git -C "{REPO_ROOT}" pull origin main --no-rebase --no-edit

sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data"
PDF_DIR  = REPO_ROOT / "pdfs"

RESULTS_DIR      = REPO_ROOT / "experiments" / "results"
TREE_DIR         = RESULTS_DIR / "vectorless_rag_index"          # one tree JSON per document
NAVIGATION_PATH  = RESULTS_DIR / "vectorless_rag_stage_navigation.jsonl"
GENERATION_PATH  = RESULTS_DIR / "vectorless_rag_stage_generation.jsonl"
SCORING_PATH     = RESULTS_DIR / "vectorless_rag_stage_scoring.jsonl"
COSTS_PATH       = RESULTS_DIR / "vectorless_rag_costs.jsonl"
INDEXING_LATENCY_PATH = RESULTS_DIR / "vectorless_rag_indexing_latency.jsonl"  # per-document wall time, method-tagged (classic vs flash)
LATENCY_PATH     = RESULTS_DIR / "vectorless_rag_latency.jsonl"

# override=True: without it, re-running this cell after editing .env keeps the
# stale value already sitting in the kernel's os.environ from an earlier run.
load_dotenv(REPO_ROOT / ".env", override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
GROQ_API_KEY   = os.getenv("GROQ_API_KEY", "")
print("GOOGLE_API_KEY:", "ok" if GOOGLE_API_KEY else "MISSING - fill in .env")
print("GROQ_API_KEY  :", "ok" if GROQ_API_KEY else "MISSING - fill in .env (needed for Stage 6's LLM judge)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already present at /content/drive/MyDrive/financebench_project — syncing to latest main ...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 31 (delta 13), reused 31 (delta 13), pack-reused 0 (from 0)
Unpacking objects: 100% (31/31), 1.37 MiB | 184.00 KiB/s, done.
From https://github.com/shaliqsv/financebench-rag-thesis
   0bf3449..ad98063  main       -> origin/main
Already on 'main'
Your branch is behind 'origin/main' by 2 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/shaliqsv/financebench-rag-thesis
 * branch            main       -> FETCH_HEAD
Updating 0bf3449..ad98063
Fast-forward
 evaluation/answer_scorer.py                        |    2 +-
 experiments/results/long_context_costs.jsonl       

In [ ]:
GOOGLE_API_KEY

In [3]:
# pinned, not "latest pageindex" -- so this notebook keeps behaving the same
# way weeks from now instead of picking up whatever VectifyAI ships next.
# pycryptodome: PyPDF2 needs it to reaad AES-encrypted PDFs (hit this on
# ADOBE_2022_10K -- some SEC filings are encrypted, not just password-locked).
%pip install -q python-dotenv pandas litellm PyPDF2 pyyaml pageindex==0.2.12 nest_asyncio groq pycryptodome

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.2/561.2 kB 15.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 76.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import litellm
import nest_asyncio
from pageindex import ConfigLoader, get_page_tokens, create_node_mapping, llm_completion, page_index_main

litellm.drop_params = True

# page_index_main runs its own asyncio.run() internally, but this notebook's
# kernel is already inside a running event loop -- nest_asyncio patches the
# loop so a nested asyncio.run() is allowed instead of raising RuntimeError.
nest_asyncio.apply()

# litellm routes by prefix ("gemini/...") and its Gemini provider reads
# GOOGLE_API_KEY natively -- no extra wiring needed.
MODEL = "gemini/gemini-3.1-flash-lite"
opt = ConfigLoader().load({"model": MODEL})
opt

namespace(toc_check_page_num=20,
          max_page_num_each_node=10,
          max_token_num_each_node=20000,
          if_add_node_id='yes',
          if_add_node_summary='yes',
          if_add_doc_description='no',
          if_add_node_text='no',
          model='gemini/gemini-3.1-flash-lite',
          index_model='gemini/gemini-3.1-flash-lite',
          summary_model='gemini/gemini-3.1-flash-lite',
          chat_model='gemini/gemini-3.1-flash-lite',
          retrieve_model='gemini/gemini-3.1-flash-lite')

---
## Cost tracking

Logs every LLM call's token counts (and cost, since `gemini-3.1-flash-lite`'s
price is filled in) to `COSTS_PATH`. `llm_completion`/`llm_acompletion` are
patched in two places: once for this notebook's own calls
(`navigate_tree`/`generate_answer`), and once inside `pageindex.page_index_classic`
— the module `page_index_main` calls internally for TOC detection,
verification, and summaries during Stage 2. Patching only the first wouldn't
see any of Stage 2's indexing cost, since that module imported its own copy
of these functions at import time (`from .utils import *`), so it isn't
looking at the same name our own `from pageindex import llm_completion`
bound.

Token counts are estimated locally via `litellm.token_counter` on the
prompt/response text, since `llm_completion` only returns the generated
text, not a usage object — not billing-exact, but the same approach the
walkthrough already validated for page-token counting. Stage 6's judge cost
is exact instead (Groq's API returns real usage).

In [4]:
import asyncio

from evaluation.cost_tracker import CostTracker, PRICING_PER_MILLION_TOKENS

cost_tracker = CostTracker(COSTS_PATH)

_current_doc_name = None
_current_financebench_id = None
_current_stage = None

_raw_llm_completion = getattr(llm_completion, "_raw", llm_completion)  # idempotent: reuse the true original even if this cell reruns


def _log_llm_usage(model, prompt, response, latency_sec):
    # response is a plain string normally, but PageIndex sometimes calls
    # llm_completion(..., return_finish_reason=True), which returns a
    # (content, finish_reason) tuple instead -- litellm.token_counter only
    # accepts a string or list of strings, so pull the text out first.
    content = response[0] if isinstance(response, tuple) else response
    input_tokens = litellm.token_counter(model=model, text=prompt)
    output_tokens = litellm.token_counter(model=model, text=content)
    cost_tracker.log(
        pipeline="vectorless_rag",
        stage=_current_stage or "unknown",
        model=model.removeprefix("gemini/"),  # matches PRICING_PER_MILLION_TOKENS' key style
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        doc_name=_current_doc_name,
        financebench_id=_current_financebench_id,
        latency_sec=latency_sec,
    )


# Redefining llm_completion at the top level makes navigate_tree/generate_answer
# use this wrapped version automatically -- they look up the bare name
# "llm_completion" fresh on every call, not the object that existed when they
# were defined. Timed here too, so every call -- indexing or per-query --
# gets its own latency logged next to its own cost, not just a per-document
# or per-question total.
import collections
import time 
REQUESTS_PER_MINUTE = 10
_call_times = collections.deque()

_rate_limit_lock = asyncio.Lock()

async def _await_rate_limit():
    async with _rate_limit_lock:
        now = time.time()
        while _call_times and now - _call_times[0] > 60:
            _call_times.popleft()
        if len(_call_times) >= REQUESTS_PER_MINUTE:
            await asyncio.sleep(60 - (now - _call_times[0]))
        _call_times.append(time.time())


def _wait_for_rate_limit():
    now = time.time()
    while _call_times and now - _call_times[0] > 60:
        _call_times.popleft()
    if len(_call_times) >= REQUESTS_PER_MINUTE:
        time.sleep(60 - (now - _call_times[0]))
    _call_times.append(time.time())


def llm_completion(model, prompt, *args, **kwargs):
    _wait_for_rate_limit()
    t0 = time.time()
    response = _raw_llm_completion(model, prompt, *args, **kwargs)
    _log_llm_usage(model, prompt, response, time.time() - t0)
    return response

llm_completion._raw = _raw_llm_completion


# PageIndex's own tree-building calls use its own copy of llm_completion/
# llm_acompletion (imported at module load time) -- patching the module's
# names directly is the only way to see that cost.
import pageindex.page_index_classic as _pageindex_internal

_raw_llm_acompletion = getattr(_pageindex_internal.llm_acompletion, "_raw", _pageindex_internal.llm_acompletion)  # idempotent: reuse the true original even if this cell reruns
_pageindex_internal.llm_completion = llm_completion

# PageIndex's node-summary step (generate_summaries_for_structure, what
# page_index_main actually calls) fires every section's summary call at once
# via asyncio.gather with no concurrency cap of its own -- a large filing with
# many sections means dozens of simultaneous but hwarequests in one burst, which is
# exactly the kind of load that trips a transient 503 from Gemini. Throttling
# here, at the one point every async call already passes through, caps that
# burst without needing to touch PageIndex's own code. Lower = fewer 503s but
# slower; higher = faster but more overload risk.
LLM_ACOMPLETION_CONCURRENCY = 10
_llm_semaphore = asyncio.Semaphore(LLM_ACOMPLETION_CONCURRENCY)


# PageIndex's own retry loop (inside _raw_llm_acompletion) only retries on a
# raised exception -- it treats a "successful" call that comes back with
# empty content as done, not failed. Gemini does occasionally return
# finish_reason="stop" with an empty message, especially under concurrent
# load, and that's what took out every node in a document at once with no
# error or retry logged anywhere: nothing ever raised, so nothing ever
# retried. This wrapper adds the retry PageIndex's own code is missing,
# specifically for that empty-but-"successful" case.
_EMPTY_RESPONSE_RETRIES = 3

async def _tracked_llm_acompletion(model, prompt):
    for attempt in range(_EMPTY_RESPONSE_RETRIES):
        t0 = time.time()
        async with _llm_semaphore:
            await _await_rate_limit()
            response = await _raw_llm_acompletion(model, prompt)
        _log_llm_usage(model, prompt, response, time.time() - t0)
        content = response[0] if isinstance(response, tuple) else response
        if content:
            return response
        if attempt < _EMPTY_RESPONSE_RETRIES - 1:
            print(f"    (empty response from {model}, retrying {attempt + 1}/{_EMPTY_RESPONSE_RETRIES}) ...")
            await asyncio.sleep(1)
    return response


_tracked_llm_acompletion._raw = _raw_llm_acompletion
_pageindex_internal.llm_acompletion = _tracked_llm_acompletion


---
## Stage 1 — Load FinanceBench data

All 150 questions across 84 unique source documents (some documents have multiple questions).

In [5]:
df_questions = pd.read_json(DATA_DIR / "financebench_open_source.jsonl", lines=True)
df_meta      = pd.read_json(DATA_DIR / "financebench_document_information.jsonl", lines=True)
df           = pd.merge(df_questions, df_meta, on=["doc_name", "company"])

doc_names = sorted(df.doc_name.unique())

print(f"Total questions : {len(df)}")
print(f"Unique documents: {len(doc_names)}")
df[["financebench_id", "doc_name", "question"]].head(3)

Total questions : 150
Unique documents: 84


,financebench_id,doc_name,question
0,financebench_id_03029,3M_2018_10K,What is the FY2018 capital expenditure amount ...
1,financebench_id_04672,3M_2018_10K,Assume that you are a public equities analyst....
2,financebench_id_00499,3M_2022_10K,Is 3M a capital-intensive business based on FY...


---
## Stage 2 — Build the PageIndex tree for every document

Parses each PDF into a hierarchical section tree once (titles + node_ids +
LLM-written summaries) and separately caches each page's raw text, so Stage 4
can pull a selected node's text later without re-parsing the PDF. Saved to
`TREE_DIR/{doc_name}_tree.json` — **resumable**, a document already indexed
is skipped.

In [6]:
import time

# Read back the results file saved so far and pull out which question IDs
# are already done, so the batch loops below know what to skip.
def _load_jsonl(path) -> list[dict]:
    if not path.exists():
        return []
    with path.open() as f:
        return [json.loads(line) for line in f if line.strip()]


def _load_completed_ids(path) -> set:
    return {r["financebench_id"] for r in _load_jsonl(path)}


# Find, check for, and read back one document's saved tree file -- needed
# because trees now get saved to disk instead of just staying in memory.
def tree_path(tree_dir, doc_name):
    return tree_dir / f"{doc_name}_tree.json"


def is_tree_built(tree_dir, doc_name) -> bool:
    return tree_path(tree_dir, doc_name).exists()


def load_tree(tree_dir, doc_name) -> dict:
    return json.loads(tree_path(tree_dir, doc_name).read_text())


# Total wall-clock time to build one document's tree, method-tagged so
# classic and flash indexing runs stay comparable -- separate from the
# per-call latency the cost tracker logs below, since the concurrency cap on
# LLM calls (see LLM_ACOMPLETION_CONCURRENCY) means a document's total time
# isn't just the sum of its calls' individual latencies.
def _log_indexing_latency(doc_name, method, elapsed_sec, n_pages, n_top_level_sections):
    INDEXING_LATENCY_PATH.parent.mkdir(parents=True, exist_ok=True)
    with INDEXING_LATENCY_PATH.open("a") as f:
        f.write(json.dumps({
            "doc_name": doc_name,
            "method": method,
            "elapsed_sec": elapsed_sec,
            "n_pages": n_pages,
            "n_top_level_sections": n_top_level_sections,
        }) + "\n")


In [8]:
# Walkthrough's Stage 1, exactly: build the tree, cache the page text.
# Wrapped in a function because this now runs once per document (84 times)
# instead of once for a single example. Timed end to end (TOC detection +
# verification + summaries + the PDF page-text cache) and logged to
# INDEXING_LATENCY_PATH, method="classic" -- the project brief's
# # preprocessing-latency-reported-separately metric.
# def build_tree(pdf_path, doc_name, opt) -> dict:
#     t0 = time.time()
#     result = page_index_main(str(pdf_path), opt)
#     page_list = get_page_tokens(str(pdf_path), model=opt.model)
#     elapsed = time.time() - t0
#     record = {
#         "doc_name": doc_name,
#         "structure": result["structure"],
#         "page_texts": [p[0] for p in page_list],
#         "n_pages": len(page_list),
#     }
#     _log_indexing_latency(doc_name, "classic", elapsed, record["n_pages"], len(record["structure"]))
#     return record


In [9]:
# # Which documents Stage 2 didn't leave a tree file for -- these were either
# # never attempted yet or hit a FAILED in the loop above and got skipped.
# missing_tree_docs = [d for d in doc_names if not is_tree_built(TREE_DIR, d)]

# print(f"{len(doc_names) - len(missing_tree_docs)}/{len(doc_names)} documents indexed")
# if missing_tree_docs:
#     print(f"Missing ({len(missing_tree_docs)}):")
#     for d in missing_tree_docs:
#         print(f"  {d}")
# else:
#     print("All documents indexed.")

---
## Stage 2b — Rebuild every document with PageIndex Flash

The classic run above (`page_index_main`) fails outright on some documents
and never even reached others: `page_index_main`'s TOC step asks the LLM to
return a strict, exactly-matching JSON table of contents, and
`gemini-3.1-flash-lite` isn't reliable enough at that strict a format to pull
it off consistently (`LLM returned a different number of TOC entries`,
`LLM returned reordered or modified TOC entries`, `FAILED: 'physical_index'`),
and a further 21 documents were never even attempted before the run got cut
off (likely a Colab disconnect).

**PageIndex Flash** (`pageindex.flash.page_index_flash`, shipped in the same
`pageindex==0.2.12` package and now the library's own documented default —
its own benchmark is run on Flash, not the classic path) sidesteps the bug
entirely: the tree structure comes from the PDF's own layout info (fonts,
headings, embedded bookmarks), not an LLM guessing a TOC in strict JSON, so
there is no strict-format-matching step left to fail. An LLM is only asked
for per-node *summaries* afterward — a far easier, less failure-prone task.

**Runs against all 84 documents, saved to a separate folder —
`TREE_DIR_FLASH`, not `TREE_DIR`.** This intentionally re-indexes the 60+
documents already done the classic way too, so both trees are built the same
way end to end and comparable — nothing is read from or written to `TREE_DIR`,
so the classic run's output is completely untouched no matter what happens
here. That does mean paying for indexing all 84 documents again — this is a
second full Stage 2 pass, not a top-up of the missing ones.

Downstream stages (3 onward) still read from `TREE_DIR` — pointing them at
`TREE_DIR_FLASH` instead is a separate step once these trees are checked over.

In [7]:
import sys

# Full run (build_all_trees_flash, 84 docs) hit 'maximum recursion depth
# exceeded' partway through -- not on any specific document (an isolated
# single-doc call to build_tree_flash on one of the 'failing' PDFs worked
# fine), so it looks like something cumulative across many page_index_flash()
# calls in one process rather than a bad PDF -- nest_asyncio.apply() (cell 4)
# lets page_index_flash's internal asyncio.run() nest inside this kernel's
# already-running loop once per document, a known way for nested-asyncio
# setups to build up call-stack depth across iterations instead of
# unwinding it. Raising the ceiling well above the default 1000 is a
# stopgap so the loop survives, not a fix for the underlying nesting.
sys.setrecursionlimit(5000)

from pageindex.flash import page_index_flash
from pageindex.utils import write_node_id

# Flash's own structure extraction never calls an LLM, but its node-summary
# step (summarize_tree, in pageindex/utils.py) and its tree-optimize step
# (pageindex/tree_optimize.py) both do -- and neither module is the
# page_index_classic module already patched above, so their calls would be
# invisible to cost_tracker without patching them too. Each module imported
# its own copy of llm_acompletion at import time (same reason the classic
# patch above targets page_index_classic specifically, not utils directly),
# so both need their own module-level name repointed -- reusing the exact
# same _tracked_llm_acompletion wrapper (same semaphore, same cost log)
# defined above rather than building a second one.
import pageindex.utils as _pageindex_utils
import pageindex.tree_optimize as _pageindex_tree_optimize

_pageindex_utils.llm_acompletion = _tracked_llm_acompletion
_pageindex_tree_optimize.llm_acompletion = _tracked_llm_acompletion

# Flash's PDF-layout parser (extract_toc, called by page_index_flash) hands
# off to a pool of worker *processes* for any document with 64+ pages --
# nearly every 10-K/10-Q here clears that. Spawning worker processes from
# inside a notebook kernel (Colab included) is a known way for this to hang
# silently and indefinitely: no error, no output, the cell just never
# returns. PageIndex's own fallback only catches a pool that raises an
# error -- not one that just hangs, which is what happened here.
#
# extract_toc's own `workers=1` argument forces its single-process path
# instead, but page_index_flash() never exposes that argument, so the only
# way to set it is patching the function main.py actually calls. main.py did
# `from .parser_pdfium_parallel import parse_charlevel_meta_parallel` --an
# explicit import, same reason as the llm_acompletion patches above -- so
# `pageindex.flash.main`, not `parser_pdfium_parallel`, is the module that
# needs its copy of the name repointed.
import pageindex.flash.main as _pageindex_flash_main

_raw_parse_charlevel_meta_parallel = getattr(
    _pageindex_flash_main.parse_charlevel_meta_parallel, "_raw", _pageindex_flash_main.parse_charlevel_meta_parallel
)  # idempotent: reuse the true original even if this cell reruns


def _parse_charlevel_meta_sequential_only(doc_handle, workers=None):
    return _raw_parse_charlevel_meta_parallel(doc_handle, workers=1)


_parse_charlevel_meta_sequential_only._raw = _raw_parse_charlevel_meta_parallel
_pageindex_flash_main.parse_charlevel_meta_parallel = _parse_charlevel_meta_sequential_only

TREE_DIR_FLASH = RESULTS_DIR / "vectorless_rag_index_flash"  # separate from TREE_DIR -- keeps the classic run's trees untouched

In [8]:
# summary=True asks the model for per-node summaries (matching classic's
# if_add_node_summary='yes'); optimize="merge" runs Flash's deterministic
# node-merging pass but skips its optional LLM "expand" pass -- expand tunes
# retrieval quality further but classic's page_index_main never did anything
# equivalent either, so leaving it off keeps this a fairer comparison and
# one fewer LLM call per document.
#
# summary_concurrency=8 caps simultaneous summary calls per document. The
# library default (64) fires every node at Gemini at once -- that burst is
# what triggered the 503 "high demand" pileup that failed 3M_2018_10K's
# whole tree (10 retries x 64 nodes, 1s apart, no backoff). 8 spreads the
# same work out instead of bursting it.
#
# page_index_flash() doesn't assign node_id itself (only its embedded-bookmark
# path does) -- write_node_id() is what page_index_main() calls internally
# for the same purpose, so calling it here keeps node_ids compatible with
# create_node_mapping/navigate_tree/fetch_node_text exactly as before.
#
# Timed end to end (structure extraction + summaries) and logged t
# INDEXING_LATENCY_PATH, method="flash" -- same metric as classic's
# build_tree, so the two are directly comparable.
def build_tree_flash(pdf_path, doc_name, model) -> dict:
    t0 = time.time()
    result = page_index_flash(str(pdf_path), summary=True, summary_model=model, optimize="merge", summary_concurrency=8)
    structure = result["structure"]
    write_node_id(structure)
    page_list = get_page_tokens(str(pdf_path), model=model)
    elapsed = time.time() - t0
    record = {
        "doc_name": doc_name,
        "structure": structure,
        "page_texts": [p[0] for p in page_list],
        "n_pages": len(page_list),
    }
    _log_indexing_latency(doc_name, "flash", elapsed, record["n_pages"], len(record["structure"]))
    return record


In [11]:
# Same loop shape as build_all_trees above, pointed at TREE_DIR_FLASH and
# run over every document (doc_names), not just the missing ones -- an
# independent, complete Stage 2 pass. Resumable, same as the classic loop:
# safe to re-run after a disconnect without re-paying for documents already
# done here.
def build_all_trees_flash(doc_names, pdf_dir, tree_dir, model):
    global _current_doc_name, _current_stage
    tree_dir.mkdir(parents=True, exist_ok=True)
    for i, doc_name in enumerate(doc_names, start=1):
        if is_tree_built(tree_dir, doc_name):
            print(f"[{i}/{len(doc_names)}] {doc_name}: already indexed (flash), skipping")
            continue
        print(f"[{i}/{len(doc_names)}] {doc_name}: building tree (flash) ...")
        _current_doc_name, _current_stage = doc_name, "indexing_flash"
        try:
            record = build_tree_flash(pdf_dir / f"{doc_name}.pdf", doc_name, MODEL)
            tree_path(tree_dir, doc_name).write_text(json.dumps(record))
            print(f"    done -- {record['n_pages']} pages, {len(record['structure'])} top-level sections")
        except Exception as e:
            print(f"    FAILED: {e}")


build_all_trees_flash(doc_names, PDF_DIR, TREE_DIR_FLASH, MODEL)

[1/84] 3M_2018_10K: already indexed (flash), skipping
[2/84] 3M_2022_10K: already indexed (flash), skipping
[3/84] 3M_2023Q2_10Q: already indexed (flash), skipping
[4/84] ACTIVISIONBLIZZARD_2019_10K: already indexed (flash), skipping
[5/84] ADOBE_2015_10K: already indexed (flash), skipping
[6/84] ADOBE_2016_10K: already indexed (flash), skipping
[7/84] ADOBE_2017_10K: already indexed (flash), skipping
[8/84] ADOBE_2022_10K: already indexed (flash), skipping
[9/84] AES_2022_10K: already indexed (flash), skipping
[10/84] AMAZON_2017_10K: already indexed (flash), skipping
[11/84] AMAZON_2019_10K: already indexed (flash), skipping
[12/84] AMCOR_2020_10K: already indexed (flash), skipping
[13/84] AMCOR_2022_8K_dated-2022-07-01: already indexed (flash), skipping
[14/84] AMCOR_2023Q2_10Q: already indexed (flash), skipping
[15/84] AMCOR_2023Q4_EARNINGS: already indexed (flash), skipping
[16/84] AMCOR_2023_10K: already indexed (flash), skipping
[17/84] AMD_2015_10K: already indexed (flash), ski

In [13]:
# Same shape as the missing_tree_docs check earlier, but against
# TREE_DIR_FLASH and all 84 documents -- how the Flash rebuild did overall,
# independent of how the classic run (still sitting untouched in TREE_DIR) did.
missing_tree_docs_flash = [d for d in doc_names if not is_tree_built(TREE_DIR_FLASH, d)]

print(f"{len(doc_names) - len(missing_tree_docs_flash)}/{len(doc_names)} documents indexed via Flash")
if missing_tree_docs_flash:
    print(f"Missing ({len(missing_tree_docs_flash)}):")
    for d in missing_tree_docs_flash:
        print(f"  {d}")
else:
    print("All documents indexed via Flash.")

84/84 documents indexed via Flash
All documents indexed via Flash.


In [14]:
# # Compares classic vs Flash indexing on the two axes that matter for the
# # thesis' cost-latency-accuracy frontier: preprocessing cost (from
# # COSTS_PATH, stage="indexing" for classic vs "indexing_flash" for Flash)
# # and preprocessing latency (from INDEXING_LATENCY_PATH, method-tagged).
# import statistics

# def summarize_indexing(latency_path, cost_path):
#     latency_records = _load_jsonl(latency_path)
#     costs = _load_jsonl(cost_path)
#     for method, stage_name in [("classic", "indexing"), ("flash", "indexing_flash")]:
#         durations = [r["elapsed_sec"] for r in latency_records if r["method"] == method]
#         stage_costs = [r["cost_usd"] for r in costs if r["stage"] == stage_name and r["cost_usd"] is not None]
#         n_calls = sum(1 for r in costs if r["stage"] == stage_name)
#         print(f"{method} ({stage_name}):")
#         print(f"  documents timed   : {len(durations)}")
#         if durations:
#             print(f"  median doc time   : {statistics.median(durations):.1f}s")
#         print(f"  LLM calls logged  : {n_calls}")
#         if stage_costs:
#             print(f"  total cost        : ${sum(stage_costs):.4f}")
#         print()


# summarize_indexing(INDEXING_LATENCY_PATH, COSTS_PATH)

In [15]:
# Prints one document's Flash tree as an indented outline -- title, page
# range, node_id, and a truncated summary per section -- so the structure
# can be sanity-checked without opening the raw JSON by hand.
def print_tree(structure, indent=0):
    for node in structure:
        pages = f"p.{node['start_index']}-{node['end_index']}"
        print(f"{'    ' * indent}[{node.get('node_id', '?')}] {node['title']} ({pages})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)


built_docs_flash = [d for d in doc_names if is_tree_built(TREE_DIR_FLASH, d)]
print(f"{len(built_docs_flash)} Flash trees on disk\n")

if built_docs_flash:
    example_doc = built_docs_flash[0]  # change to any doc_name in built_docs_flash
    print(f"=== {example_doc} ===")
    g = load_tree(TREE_DIR_FLASH, example_doc)
    print_tree(g["structure"])


84 Flash trees on disk

=== 3M_2018_10K ===
[0000] Cover Page (p.1-1)
[0001] Table of Contents (p.2-3)
[0002] Item 1. Business (p.4-9)
[0003] Item 1A. Risk Factors (p.10-12)
[0004] Item 1B. Unresolved Staff Comments (p.12-12)
[0005] Item 2. Properties (p.12-12)
[0006] Item 3. Legal Proceedings (p.12-12)
[0007] Item 4. Mine Safety Disclosures (p.12-12)
[0008] Item 5. Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities (p.13-13)
[0009] Item 6. Selected Financial Data (p.14-14)
[0010] Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations (p.15-51)
[0011] Item 7A. Quantitative and Qualitative Disclosures About Market Risk (p.51-52)
[0012] Item 8. Financial Statements and Supplementary Data (p.52-127)
[0013] Item 9. Changes in and Disagreements with Accountants on Accounting and Financial Disclosure (p.128-128)
[0014] Item 9A. Controls and Procedures (p.128-128)
[0015] Item 9B. Other Information

---
## Coverage check -- catch trees that silently skip a document's opening pages

`3M_2018_10K` and `3M_2022_10K` were built before the parallel-parser hang was patched
to force the sequential path (`parse_charlevel_meta_parallel` -> `workers=1`, above). If
those two trees predate that fix, they may have been built from a run where the worker
pool silently dropped the early pages instead of hanging -- `is_tree_built` only checks
that a tree *file* exists, not that it actually covers the whole document, so a partial
tree like that looks indexed and never gets flagged or retried.

This checks each built tree's top-level page ranges against `n_pages` and flags any gap,
then rebuilds whatever fails the check. `financebench_id_03029` (a 3M_2018_10K question,
seen in the query-expansion diff above) picking almost every node in the tree is the kind
of symptom a missing early section produces -- no node actually covers the right page, so
the model has nothing good to converge on.


In [16]:
import datetime

# Sanity check before rebuilding anything -- confirms these two trees are in
# fact old enough to predate the workers=1 fix, not just coincidentally broken.
for doc_name in ["3M_2018_10K", "3M_2022_10K"]:
    p = tree_path(TREE_DIR_FLASH, doc_name)
    if p.exists():
        print(f"{doc_name}: tree file last modified {datetime.datetime.fromtimestamp(p.stat().st_mtime)}")
    else:
        print(f"{doc_name}: no tree file found")


3M_2018_10K: tree file last modified 2026-09-18 12:14:56
3M_2022_10K: tree file last modified 2026-09-18 12:19:14


In [21]:
# Flags any built tree whose top-level sections don't fully span pages
# 1..n_pages -- a gap means some part of the document was never turned into a
# node at all, so navigation can never select it no matter how good the query is.
def tree_coverage_gaps(tree_dir, doc_name):
    tree = load_tree(tree_dir, doc_name)
    structure, n_pages = tree["structure"], tree["n_pages"]
    if not structure:
        return {"doc_name": doc_name, "n_pages": n_pages, "gaps": ["EMPTY STRUCTURE"]}

    ranges = sorted((n["start_index"], n["end_index"]) for n in structure)
    gaps = []
    if ranges[0][0] != 1:
        gaps.append(f"missing pages 1-{ranges[0][0] - 1}")
    for (s1, e1), (s2, e2) in zip(ranges, ranges[1:]):
        if s2 > e1 + 1:
            gaps.append(f"missing pages {e1 + 1}-{s2 - 1}")
    if ranges[-1][1] != n_pages:
        gaps.append(f"missing pages {ranges[-1][1] + 1}-{n_pages}")
    return {"doc_name": doc_name, "n_pages": n_pages, "covers": f"{ranges[0][0]}-{ranges[-1][1]}", "gaps": gaps}


coverage_df = pd.DataFrame(tree_coverage_gaps(TREE_DIR_FLASH, d) for d in built_docs_flash)
bad_coverage = coverage_df[coverage_df["gaps"].apply(len) > 0]
print(f"{len(bad_coverage)}/{len(coverage_df)} documents have coverage gaps")
bad_coverage


22/84 documents have coverage gaps


,doc_name,n_pages,covers,gaps
15,AMCOR_2023_10K,156,2-156,[missing pages 1-1]
24,BESTBUY_2023_10K,75,2-75,[missing pages 1-1]
26,BLOCK_2016_10K,121,3-121,[missing pages 1-2]
27,BLOCK_2020_10K,158,4-158,[missing pages 1-3]
31,COCACOLA_2021_10K,183,2-183,[missing pages 1-1]
32,COCACOLA_2022_10K,183,2-183,[missing pages 1-1]
38,CVSHEALTH_2022_10K,213,3-213,[missing pages 1-2]
39,FOOTLOCKER_2022_8K_dated-2022-05-20,4,2-4,[missing pages 1-1]
40,FOOTLOCKER_2022_8K_dated_2022-08-19,31,4-31,[missing pages 1-3]
42,GENERALMILLS_2020_10K,127,2-127,[missing pages 1-1]


In [17]:

import re

def total_gap_pages(gaps):
    total = 0
    for g in gaps:
        m = re.search(r"(\d+)-(\d+)", g)
        if m:
            total += int(m.group(2)) - int(m.group(1)) + 1
    return total


bad_coverage["gap_pages"] = bad_coverage["gaps"].apply(total_gap_pages)
docs_to_rebuild = bad_coverage.loc[bad_coverage["gap_pages"] > 3, "doc_name"].tolist()
print(f"rebuilding {len(docs_to_rebuild)}/{len(bad_coverage)} documents (gap > 3 pages): {docs_to_rebuild}")

bad_coverage[["doc_name", "gap_pages", "gaps"]].sort_values("gap_pages", ascending=False)

rebuilding 16/38 documents (gap > 3 pages): ['AMERICANWATERWORKS_2022_10K', 'BESTBUY_2017_10K', 'BESTBUY_2019_10K', 'BESTBUY_2024Q2_10Q', 'CORNING_2021_10K', 'CORNING_2022_10K', 'COSTCO_2021_10K', 'GENERALMILLS_2019_10K', 'KRAFTHEINZ_2019_10K', 'LOCKHEEDMARTIN_2020_10K', 'LOCKHEEDMARTIN_2021_10K', 'MICROSOFT_2016_10K', 'NIKE_2018_10K', 'NIKE_2019_10K', 'NIKE_2021_10K', 'ULTABEAUTY_2023_10K']


/tmp/ipykernel_9539/12449485.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bad_coverage["gap_pages"] = bad_coverage["gaps"].apply(total_gap_pages)


,doc_name,gap_pages,gaps
21,AMERICANWATERWORKS_2022_10K,144,[missing pages 1-144]
35,CORNING_2022_10K,86,[missing pages 1-86]
54,LOCKHEEDMARTIN_2021_10K,79,[missing pages 1-79]
53,LOCKHEEDMARTIN_2020_10K,71,[missing pages 1-71]
41,GENERALMILLS_2019_10K,70,[missing pages 1-70]
34,CORNING_2021_10K,63,[missing pages 1-63]
61,MICROSOFT_2016_10K,61,[missing pages 1-61]
78,ULTABEAUTY_2023_10K,60,[missing pages 1-60]
22,BESTBUY_2017_10K,59,[missing pages 1-59]
66,NIKE_2019_10K,57,[missing pages 1-57]


---
## LLM table-of-contents fallback -- for docs Flash's rebuild still can't fix

`tree_coverage_gaps` found this isn't a rebuild-fixable bug: PageIndex Flash's font/layout
heading heuristic deterministically finds zero headings before page 85 in `3M_2018_10K`
(verified by tracing the library's own heading-collection step locally) -- rebuilding the
same PDF with the same heuristic gives the same result every time.

`pipelines/vectorless_rag/llm_toc_fallback.py` replaces heading detection with a single LLM
call (reads meaning, not font size, so it isn't fooled by this filing's typography) for just
the documents that still fail the coverage check after the Flash rebuild above. Everything
else -- navigation, generation, retrieval scoring -- is untouched and still runs the normal
PageIndex pipeline for every other document.


In [9]:
from pipelines.vectorless_rag.llm_toc_fallback import build_tree_llm_fallback


def rebuild_with_llm_fallback(doc_names, pdf_dir, tree_dir, model):
    global _current_doc_name, _current_stage
    for doc_name in doc_names:
        print(f"{doc_name}: building TOC via LLM fallback ...")
        _current_doc_name, _current_stage = doc_name, "indexing_fallback"
        page_list = get_page_tokens(str(pdf_dir / f"{doc_name}.pdf"), model=model)
        page_texts = [p[0] for p in page_list]
        record = build_tree_llm_fallback(doc_name, page_texts, model, llm_completion)
        tree_path(tree_dir, doc_name).write_text(json.dumps(record))
        print(f"    done -- {record['n_pages']} pages, {len(record['structure'])} sections")



In [ ]:

# Only the ones the Flash rebuild above didn't actually fix -- most flagged
# docs might just have been the delete-before-rebuild mishap (already solved
# by re-running build_all_trees_flash), not a real heuristic miss.
still_bad = [d for d in docs_to_rebuild if tree_coverage_gaps(TREE_DIR_FLASH, d)["gaps"]]
print(f"{len(still_bad)} documents still failing coverage after Flash rebuild: {still_bad}")

rebuild_with_llm_fallback(still_bad, PDF_DIR, TREE_DIR_FLASH, MODEL)


16 documents still failing coverage after Flash rebuild: ['AMERICANWATERWORKS_2022_10K', 'BESTBUY_2017_10K', 'BESTBUY_2019_10K', 'BESTBUY_2024Q2_10Q', 'CORNING_2021_10K', 'CORNING_2022_10K', 'COSTCO_2021_10K', 'GENERALMILLS_2019_10K', 'KRAFTHEINZ_2019_10K', 'LOCKHEEDMARTIN_2020_10K', 'LOCKHEEDMARTIN_2021_10K', 'MICROSOFT_2016_10K', 'NIKE_2018_10K', 'NIKE_2019_10K', 'NIKE_2021_10K', 'ULTABEAUTY_2023_10K']
AMERICANWATERWORKS_2022_10K: building TOC via LLM fallback ...
    done -- 306 pages, 26 sections
BESTBUY_2017_10K: building TOC via LLM fallback ...
    done -- 109 pages, 21 sections
BESTBUY_2019_10K: building TOC via LLM fallback ...
    done -- 107 pages, 31 sections
BESTBUY_2024Q2_10Q: building TOC via LLM fallback ...
    done -- 30 pages, 10 sections
CORNING_2021_10K: building TOC via LLM fallback ...
    done -- 125 pages, 14 sections
CORNING_2022_10K: building TOC via LLM fallback ...
    done -- 159 pages, 31 sections
COSTCO_2021_10K: building TOC via LLM fallback ...
    don

In [20]:
for doc_name in still_bad:
    print(f"=== {doc_name} ===")
    print_tree(load_tree(TREE_DIR_FLASH, doc_name)["structure"])
    print(tree_coverage_gaps(TREE_DIR_FLASH, doc_name))
    print()


=== AMERICANWATERWORKS_2022_10K ===
[0000] Cover Page (p.1-1)
[0001] Table of Contents (p.2-2)
[0002] Forward-Looking Statements (p.3-5)
[0003] Item 1. Business (p.6-24)
[0004] Item 1A. Risk Factors (p.25-39)
[0005] Item 1B. Unresolved Staff Comments (p.40-40)
[0006] Item 2. Properties (p.40-40)
[0007] Item 3. Legal Proceedings (p.40-46)
[0008] Item 4. Mine Safety Disclosures (p.46-46)
[0009] Item 5. Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities (p.47-47)
[0010] Item 6. [Reserved] (p.47-47)
[0011] Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations (p.48-75)
[0012] Item 8. Financial Statements and Supplementary Data (p.76-133)
[0013] Item 9. Changes in and Disagreements with Accountants on Accounting and Financial Disclosure (p.134-134)
[0014] Item 9A. Controls and Procedures (p.134-134)
[0015] Item 9B. Other Information (p.135-135)
[0016] Item 9C. Disclosure Regarding Foreign Juris

In [ ]:
# The rebuilt trees have different structure/node_ids now, so any
# navigation/generation results already saved for these documents'
# questions were computed against the OLD, broken tree -- run_navigation_all
# etc. skip a financebench_id purely because it's already in the output file,
# so those stale records need to be removed before rerunning, or the pipeline
# will just keep the bad answers instead of regenerating them.
def purge_stale_records(path, doc_names):
    if not path.exists():
        return
    records = _load_jsonl(path)
    kept = [r for r in records if r.get("doc_name") not in doc_names]
    removed = len(records) - len(kept)
    with path.open("w") as f:
        for r in kept:
            f.write(json.dumps(r) + "\n")
    print(f"{path.name}: removed {removed} stale records")


rebuilt_docs = set(docs_to_rebuild)
for path in [NAVIGATION_PATH, EXPANDED_NAVIGATION_PATH, GENERATION_PATH]:
    purge_stale_records(path, rebuilt_docs)

print("\nRe-run Stage 3 (and the query-expansion cells, and Stage 4+) to regenerate these.")


In [ ]:
import re, pandas as pd

def orphan_pages(node):
    """Pages inside a node's range that none of its children cover."""
    covered = set()
    for ch in node.get("nodes", []):
        covered.update(range(ch["start_index"], ch["end_index"] + 1))
    return [p for p in range(node["start_index"], node["end_index"] + 1) if p not in covered]

MDNA_RE = re.compile(r"management.s discussion|\bitem\s*7\b", re.I)
FIN_RE  = re.compile(r"financial statements|\bitem\s*8\b|\bitem\s*15\b|balance sheet|statements? of (income|operations)", re.I)

def audit_trees(tree_dir, doc_names, min_orphans=15):
    rows = []
    for d in doc_names:
        if not is_tree_built(tree_dir, d):
            continue
        tree = load_tree(tree_dir, d)
        titles, worst = [], {"orphans": 0, "title": ""}

        def walk(nodes):
            for n in nodes:
                titles.append(n["title"])
                if n.get("nodes"):
                    k = len(orphan_pages(n))
                    if k > worst["orphans"]:
                        worst.update(orphans=k, title=n["title"][:50])
                    walk(n["nodes"])
        walk(tree["structure"])

        is_filing = d.endswith(("10K", "10Q"))
        no_mdna = is_filing and not any(MDNA_RE.search(t) for t in titles)
        no_fin  = is_filing and not any(FIN_RE.search(t) for t in titles)
        rows.append({
            "doc": d, "n_pages": tree["n_pages"],
            "worst_orphans": worst["orphans"], "worst_title": worst["title"],
            "no_mdna_title": no_mdna, "no_financials_title": no_fin,
            "FLAGGED": worst["orphans"] > min_orphans or (no_mdna and no_fin),
        })
    return pd.DataFrame(rows).sort_values("worst_orphans", ascending=False)

audit = audit_trees(TREE_DIR_FLASH, doc_names)
audit[audit["FLAGGED"]]

,doc,n_pages,worst_orphans,worst_title,no_mdna_title,no_financials_title,FLAGGED
18,AMERICANEXPRESS_2022_10K,260,130,DOCUMENTS INCORPORATED BY REFERENCE,True,True,True
3,ACTIVISIONBLIZZARD_2019_10K,198,99,Documents Incorporated by Reference,True,True,True
29,BOEING_2022_10K,190,97,DOCUMENTS INCORPORATED BY REFERENCE,True,False,True
70,PEPSICO_2021_10K,549,76,Key Employee:,True,True,True
28,BOEING_2018_10K,136,68,DOCUMENTS INCORPORATED BY REFERENCE,True,False,True
69,PAYPAL_2022_10K,134,68,DOCUMENTS INCORPORATED BY REFERENCE,True,False,True
64,NETFLIX_2017_10K,73,46,DOCUMENTS INCORPORATED BY REFERENCE,True,True,True
63,NETFLIX_2015_10K,72,44,DOCUMENTS INCORPORATED BY REFERENCE,True,True,True
81,WALMART_2018_10K,304,18,APPENDIX A,False,False,True
71,PEPSICO_2022_10K,503,15,APPENDIX,True,True,True


In [49]:
# Drop stale results so the resumable loops redo these docs

flagged_docs = audit.loc[audit["FLAGGED"], "doc"].tolist()
print(f"{len(flagged_docs)} docs to rebuild: {flagged_docs}")

# Overwrites the existing tree files for these docs
rebuild_with_llm_fallback(flagged_docs, PDF_DIR, TREE_DIR_FLASH, MODEL)

# Drop stale results so the resumable loops redo these docs
def drop_docs_from_jsonl(path, docs):
    if not path.exists():
        return 0
    records = _load_jsonl(path)
    kept = [r for r in records if r.get("doc_name") not in docs]
    path.write_text("".join(json.dumps(r) + "\n" for r in kept))
    return len(records) - len(kept)

for p in [NAVIGATION_PATH, EXPANDED_NAVIGATION_PATH, GENERATION_PATH, SCORING_PATH]:
    print(f"{p.name}: removed {drop_docs_from_jsonl(p, set(flagged_docs))} stale records")

audit_trees(TREE_DIR_FLASH, flagged_docs)

10 docs to rebuild: ['AMERICANEXPRESS_2022_10K', 'ACTIVISIONBLIZZARD_2019_10K', 'BOEING_2022_10K', 'PEPSICO_2021_10K', 'BOEING_2018_10K', 'PAYPAL_2022_10K', 'NETFLIX_2017_10K', 'NETFLIX_2015_10K', 'WALMART_2018_10K', 'PEPSICO_2022_10K']
AMERICANEXPRESS_2022_10K: building TOC via LLM fallback ...
    done -- 260 pages, 23 sections
ACTIVISIONBLIZZARD_2019_10K: building TOC via LLM fallback ...
    done -- 198 pages, 31 sections
BOEING_2022_10K: building TOC via LLM fallback ...
    done -- 190 pages, 26 sections
PEPSICO_2021_10K: building TOC via LLM fallback ...
    done -- 549 pages, 58 sections
BOEING_2018_10K: building TOC via LLM fallback ...
    done -- 136 pages, 16 sections
PAYPAL_2022_10K: building TOC via LLM fallback ...
    done -- 134 pages, 24 sections
NETFLIX_2017_10K: building TOC via LLM fallback ...
    done -- 73 pages, 26 sections
NETFLIX_2015_10K: building TOC via LLM fallback ...
    done -- 72 pages, 21 sections
WALMART_2018_10K: building TOC via LLM fallback ...
 

,doc,n_pages,worst_orphans,worst_title,no_mdna_title,no_financials_title,FLAGGED
0,AMERICANEXPRESS_2022_10K,260,0,,False,False,False
1,ACTIVISIONBLIZZARD_2019_10K,198,0,,False,False,False
2,BOEING_2022_10K,190,0,,False,False,False
3,PEPSICO_2021_10K,549,0,,False,False,False
4,BOEING_2018_10K,136,0,,False,False,False
5,PAYPAL_2022_10K,134,0,,False,False,False
6,NETFLIX_2017_10K,73,0,,False,False,False
7,NETFLIX_2015_10K,72,0,,False,False,False
8,WALMART_2018_10K,304,0,,False,False,False
9,PEPSICO_2022_10K,503,0,,False,False,False


In [ ]:
16 documents still failing coverage after Flash rebuild: ['AMERICANWATERWORKS_2022_10K', 'BESTBUY_2017_10K', 'BESTBUY_2019_10K', 'BESTBUY_2024Q2_10Q', 'CORNING_2021_10K', 'CORNING_2022_10K', 'COSTCO_2021_10K', 'GENERALMILLS_2019_10K', 'KRAFTHEINZ_2019_10K', 'LOCKHEEDMARTIN_2020_10K', 'LOCKHEEDMARTIN_2021_10K', 'MICROSOFT_2016_10K', 'NIKE_2018_10K', 'NIKE_2019_10K', 'NIKE_2021_10K', 'ULTABEAUTY_2023_10K']

---
## Stage 3 — Navigate the tree (select relevant sections)

One LLM call per question: the whole tree (titles, node_ids, summaries — no
section text) is shown at once, and the model returns a ranked JSON array of
node_ids likely to contain the answer. If parsing fails or the model returns
nothing usable, this falls back to *every* node in the tree rather than
silently generating from zero context in Stage 4.

**Resumable** — needs Stage 2 to have built the document's tree first.

In [10]:
NAVIGATION_PROMPT = """You are navigating a financial filing's table-of-contents-style section tree \
to find the section(s) most likely to contain the answer to a question. You cannot see the section \
text yet, only titles and summaries.

Financial filings often place specific figures (balance sheet items, income statement lines, \
segment data) in predictable sections like the balance sheet, income statement, cash flow \
statement, or notes to financial statements. Narrative or qualitative questions may require \
broader sections like MD&A or risk factors. Some questions require comparing figures across \
two years or two filings, which may mean the answer spans more than one section.

First, briefly reason about which sections are plausible candidates and why, based on the \
titles and summaries given. Then list the node_ids of every section that could plausibly \
contain the answer, ordered from most to least likely. Only include node_ids that appear \
exactly as given in the tree below. When uncertain whether a section is relevant, include it \
rather than exclude it. Do not artificially limit the list length, include as many or as few \
sections as the question genuinely requires.

Question: {question}

Document tree:
{tree_outline}

Respond in this exact JSON format:
{{
  "thinking": "<your reasoning about which sections are relevant and why>",
  "node_ids": ["<id1>", "<id2>", ...]
}}"""


# Walkthrough's Stage 2, exactly: build the prompt, ask the model, pull the
# node_ids out of the reply. One addition -- if the model's answer can't be
# parsed or comes back empty, fall back to every node in the tree, so Stage 4
# never generates from zero information.
def navigate_tree(question, structure, model) -> str:
    prompt = NAVIGATION_PROMPT.format(question=question, tree_outline=json.dumps(structure, indent=2))
    return llm_completion(model, prompt)


def parse_navigation_response(raw_response, structure) -> list:
    node_map = create_node_mapping(structure)
    match = re.search(r"\[.*\]", raw_response, re.DOTALL)
    node_ids = json.loads(match.group(0)) if match else []
    if not node_ids:
        node_ids = list(node_map.keys())  # fallback: whole document
    return node_ids


In [51]:
# The loop over all 150 questions: skip ones already navigated, skip ones
# whose document isn't indexed yet, otherwise call navigate_tree and save
# the result.
def run_navigation_all(df, tree_dir, out_path, model):
    global _current_doc_name, _current_financebench_id, _current_stage
    out_path.parent.mkdir(parents=True, exist_ok=True)
    completed = _load_completed_ids(out_path)
    tree_cache = {}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            continue
        if not is_tree_built(tree_dir, row.doc_name):
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED -- {row.doc_name} not indexed yet")
            continue

        print(f"[{i}/{len(df)}] {fb_id}: navigating ...")
        _current_doc_name, _current_financebench_id, _current_stage = row.doc_name, fb_id, "navigation"
        try:
            if row.doc_name not in tree_cache:
                tree_cache[row.doc_name] = load_tree(tree_dir, row.doc_name)
            structure = tree_cache[row.doc_name]["structure"]
            raw_response = navigate_tree(row.question, structure, model)
            node_ids = parse_navigation_response(raw_response, structure)
            record = {"financebench_id": fb_id, "doc_name": row.doc_name, "node_ids": node_ids, "raw_response": raw_response, "timestamp": time.time()}
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
        except Exception as e:
            print(f"    FAILED: {e}")


run_navigation_all(df, TREE_DIR_FLASH, NAVIGATION_PATH, MODEL)

[9/150] financebench_id_02987: navigating ...
[10/150] financebench_id_07966: navigating ...
[32/150] financebench_id_00222: navigating ...
[39/150] financebench_id_00476: navigating ...
[40/150] financebench_id_01028: navigating ...
[41/150] financebench_id_00723: navigating ...
[42/150] financebench_id_00720: navigating ...
[43/150] financebench_id_01351: navigating ...
[44/150] financebench_id_01964: navigating ...
[45/150] financebench_id_01981: navigating ...
[60/150] financebench_id_10285: navigating ...
[61/150] financebench_id_00517: navigating ...
[62/150] financebench_id_01091: navigating ...
[63/150] financebench_id_00678: navigating ...
[64/150] financebench_id_01290: navigating ...
[65/150] financebench_id_00464: navigating ...
[66/150] financebench_id_00494: navigating ...
[67/150] financebench_id_00585: navigating ...
[113/150] financebench_id_04458: navigating ...
[114/150] financebench_id_03282: navigating ...
[119/150] financebench_id_00080: navigating ...
[120/150] f

In [26]:
nav_records = _load_jsonl(NAVIGATION_PATH)
print(f"{len(nav_records)} questions navigated\n")
nn=pd.DataFrame([
    {"financebench_id": r["financebench_id"], "doc_name": r["doc_name"], "n_nodes_selected": len(r["node_ids"])}
    for r in nav_records
])
nn['n_nodes_selected'].describe()
nav_records

150 questions navigated



[{'financebench_id': 'financebench_id_03029',
  'doc_name': '3M_2018_10K',
  'node_ids': ['0012', '0010'],
  'raw_response': '{\n  "thinking": "The question asks for the FY2018 capital expenditure amount, specifically requesting that the answer be derived from the cash flow statement. Capital expenditures are a standard line item in the Investing Activities section of the Statement of Cash Flows. According to the document tree, \'Item 8. Financial Statements and Supplementary Data\' (node_id: \'0012\') contains the consolidated financial statements, which include the cash flow statement. \'Item 7. Management’s Discussion and Analysis\' (node_id: \'0010\') often discusses capital expenditures and liquidity, making it a secondary, highly relevant source for verifying or contextually explaining the figure found in the financial statements.",\n  "node_ids": ["0012", "0010"]\n}'},
 {'financebench_id': 'financebench_id_04672',
  'doc_name': '3M_2018_10K',
  'node_ids': ['0012', '0009'],
  'r

In [27]:
tree_cache = {}
rows = []
for r in nav_records:
    fb_id, doc_name = r["financebench_id"], r["doc_name"]
    gold_row = df.loc[df.financebench_id == fb_id].iloc[0]
    gold_pages = {e["evidence_page_num"] for e in gold_row.evidence}

    if doc_name not in tree_cache:
        tree_cache[doc_name] = load_tree(TREE_DIR_FLASH, doc_name)
    node_map = create_node_mapping(tree_cache[doc_name]["structure"])

    covered_pages = set()
    for node_id in r["node_ids"]:
        node = node_map.get(node_id)
        if node is None:
            continue
        covered_pages.update(range(node["start_index"] - 1, node["end_index"]))  # -1: pages are 1-indexed, gold is 0-indexed

    rows.append({
        "financebench_id": fb_id,
        "doc_name": doc_name,
        "n_nodes_selected": len(r["node_ids"]),
        "gold_pages": sorted(gold_pages),
        "hit": bool(gold_pages & covered_pages),
        'covered_pages': sorted(covered_pages),
    })

recall_df = pd.DataFrame(rows)
recall_df


,financebench_id,doc_name,n_nodes_selected,gold_pages,hit,covered_pages
0,financebench_id_03029,3M_2018_10K,2,[59],True,"[14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 2..."
1,financebench_id_04672,3M_2018_10K,2,[57],True,"[13, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 6..."
2,financebench_id_00499,3M_2022_10K,4,"[47, 49, 51]",True,"[3, 4, 5, 6, 7, 8, 15, 18, 19, 20, 21, 22, 23,..."
3,financebench_id_01226,3M_2022_10K,2,[26],True,"[18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 2..."
4,financebench_id_01865,3M_2022_10K,2,[24],True,"[3, 4, 5, 6, 7, 8, 18, 19, 20, 21, 22, 23, 24,..."
...,...,...,...,...,...,...
145,financebench_id_00735,PEPSICO_2022_10K,3,[25],True,"[12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 2..."
146,financebench_id_01328,PEPSICO_2022_10K,4,[77],True,"[30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4..."
147,financebench_id_03620,PEPSICO_2022_10K,2,"[61, 63]",True,"[61, 62, 63]"
148,financebench_id_04481,PEPSICO_2022_10K,3,"[61, 63]",True,"[30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4..."


In [54]:
recall_df

,financebench_id,doc_name,n_nodes_selected,gold_pages,hit,covered_pages
0,financebench_id_03029,3M_2018_10K,2,[59],True,"[14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 2..."
1,financebench_id_04672,3M_2018_10K,2,[57],True,"[13, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 6..."
2,financebench_id_00499,3M_2022_10K,4,"[47, 49, 51]",True,"[3, 4, 5, 6, 7, 8, 15, 18, 19, 20, 21, 22, 23,..."
3,financebench_id_01226,3M_2022_10K,2,[26],True,"[18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 2..."
4,financebench_id_01865,3M_2022_10K,2,[24],True,"[3, 4, 5, 6, 7, 8, 18, 19, 20, 21, 22, 23, 24,..."
...,...,...,...,...,...,...
145,financebench_id_00735,PEPSICO_2022_10K,3,[25],True,"[12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 2..."
146,financebench_id_01328,PEPSICO_2022_10K,4,[77],True,"[30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4..."
147,financebench_id_03620,PEPSICO_2022_10K,2,"[61, 63]",True,"[61, 62, 63]"
148,financebench_id_04481,PEPSICO_2022_10K,3,"[61, 63]",True,"[30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4..."


In [55]:
# Per-document hit/miss, joined with tree properties
def tree_info(doc_name):
    t = load_tree(TREE_DIR_FLASH, doc_name)
    top = t["structure"]
    return {
        "doc_name": doc_name,
        "n_pages": t["n_pages"],
        "n_top_sections": len(top),
        "avg_pages_per_top_section": round(t["n_pages"] / max(len(top), 1), 1),
        "nested": any(n.get("nodes") for n in top),   # False = flat (likely the LLM fallback)
        "has_unlabeled": any(n["title"] == "Unlabeled" for n in top),
    }

per_doc = (recall_df.groupby("doc_name")
           .agg(questions=("hit", "size"), hits=("hit", "sum"))
           .assign(misses=lambda d: d.questions - d.hits,
                   hit_rate=lambda d: (d.hits / d.questions).round(2))
           .reset_index())
per_doc = per_doc.merge(pd.DataFrame(tree_info(d) for d in per_doc.doc_name), on="doc_name")
per_doc.sort_values(["misses", "hit_rate"], ascending=[False, True])

,doc_name,questions,hits,misses,hit_rate,n_pages,n_top_sections,avg_pages_per_top_section,nested,has_unlabeled
74,PEPSICO_2023_8K_dated-2023-05-30,2,0,2,0.00,172,40,4.3,True,False
38,CVSHEALTH_2022_10K,3,1,2,0.33,213,140,1.5,True,False
40,FOOTLOCKER_2022_8K_dated_2022-08-19,1,0,1,0.00,31,4,7.8,False,False
50,JPMORGAN_2022_10K,1,0,1,0.00,382,21,18.2,True,False
82,WALMART_2019_10K,1,0,1,0.00,155,34,4.6,True,False
...,...,...,...,...,...,...,...,...,...,...
78,ULTABEAUTY_2023_10K,2,2,0,1.00,105,27,3.9,False,False
79,VERIZON_2021_10K,2,2,0,1.00,120,1,120.0,True,False
80,VERIZON_2022_10K,3,3,0,1.00,124,1,124.0,True,False
81,WALMART_2018_10K,1,1,0,1.00,304,23,13.2,False,False


In [56]:
per_doc.loc[per_doc["hit_rate"] < 0.5, ["doc_name", "n_pages", "n_top_sections", "avg_pages_per_top_section", "nested", "has_unlabeled", "questions", "hits", "misses", "hit_rate"]].sort_values("misses", ascending=False)

,doc_name,n_pages,n_top_sections,avg_pages_per_top_section,nested,has_unlabeled,questions,hits,misses,hit_rate
38,CVSHEALTH_2022_10K,213,140,1.5,True,False,3,1,2,0.33
74,PEPSICO_2023_8K_dated-2023-05-30,172,40,4.3,True,False,2,0,2,0.00
40,FOOTLOCKER_2022_8K_dated_2022-08-19,31,4,7.8,False,False,1,0,1,0.00
50,JPMORGAN_2022_10K,382,21,18.2,True,False,1,0,1,0.00
82,WALMART_2019_10K,155,34,4.6,True,False,1,0,1,0.00


In [57]:
# against what the model actually picked.
def find_node_path(structure, page, path=None):
    path = path or []
    for node in structure:
        if node["start_index"] - 1 <= page <= node["end_index"] - 1:
            new_path = path + [node]
            deeper = find_node_path(node.get("nodes", []), page, new_path)
            return deeper if deeper else new_path
    return None


def inspect_miss(fb_id):
    row = df.loc[df.financebench_id == fb_id].iloc[0]
    nav = next(r for r in nav_records if r["financebench_id"] == fb_id)
    if row.doc_name not in tree_cache:
        tree_cache[row.doc_name] = load_tree(TREE_DIR_FLASH, row.doc_name)
    tree = tree_cache[row.doc_name]
    node_map = create_node_mapping(tree["structure"])
    gold_pages = sorted({e["evidence_page_num"] for e in row.evidence})

    print(f"Q: {row.question}")
    print(f"doc: {row.doc_name}  gold pages: {gold_pages}\n")

    print("--- model's raw response (reasoning + picks) ---")
    print(nav.get("raw_response", "<not saved -- re-run navigation with raw_response logging first>"))

    print("\n--- what it picked ---")
    for nid in nav["node_ids"]:
        n = node_map.get(nid)
        if n:
            print(f"  [{nid}] {n['title']} (p.{n['start_index']}-{n['end_index']})")

    print("\n--- correct section(s) + siblings ---")
    for page in gold_pages:
        path = find_node_path(tree["structure"], page)
        if not path:
            print(f"  p.{page}: no node covers this page")
            continue
        correct = path[-1]
        siblings = path[-2]["nodes"] if len(path) > 1 else tree["structure"]
        print(f"  p.{page} is in [{correct['node_id']}] {correct['title']}")
        for sib in siblings:
            marker = "<-- CORRECT" if sib["node_id"] == correct["node_id"] else \
                     ("<-- picked" if sib["node_id"] in nav["node_ids"] else "")
            print(f"      [{sib['node_id']}] {sib['title']} {marker}")


# recall_df must already exist (hit/miss from the earlier recall check)
misses = recall_df.loc[~recall_df["hit"], "financebench_id"].tolist()
print(f"{len(misses)} missed questions\n")

for fb_id in misses[:5]:  # bump the slice to see more
    inspect_miss(fb_id)
    print("=" * 100)

11 missed questions

Q: How much has the effective tax rate of Corning changed between FY2021 and FY2022?
doc: CORNING_2022_10K  gold pages: [23]

--- model's raw response (reasoning + picks) ---
{
  "thinking": "The question asks for a comparison of the effective tax rate between 2021 and 2022. The \"Consolidated Financial Statements and Notes\" (node_id 0029) is the most critical section as it explicitly contains the financial statements and the accompanying notes, which include detailed breakdowns of income taxes and tax rate reconciliations. The \"Index to Financial Statements\" (node_id 0027) confirms that the notes cover income taxes, which will contain the specific tax rate data for the requested years.",
  "node_ids": ["0029", "0027"]
}

--- what it picked ---
  [0029] Consolidated Financial Statements and Notes (p.58-106)
  [0027] Index to Financial Statements (p.55-55)

--- correct section(s) + siblings ---
  p.23 is in [0008] Part II
      [0000] Cover Page 
      [0001] Par

---
## Stage 3.5 — Query expansion ablation (does rewriting the question help navigation?)

Same idea as Vector RAG's Stage 5 query expansion: ask the model to rewrite the question
into filing terminology before using it for retrieval. Applied here to Stage 3's navigation
call instead of embedding. Motivation: one of the two navigation failure modes seen in
`inspect_miss` above is a **terminology mismatch** -- the question uses everyday phrasing
("how much cash did they have on hand") while section titles use formal filing language
("Cash and Cash Equivalents"). Expanding the query first is a way to close that gap without
touching the navigation prompt itself.

Saved separately (`vectorless_rag_stage_navigation_expanded.jsonl`) so the original Stage 3
run stays untouched and the two are directly comparable.

**Costs 2x the LLM calls per question** (expand + navigate) -- worth knowing given the quota
limits already hit.


In [13]:
# Same rewrite-the-question idea as Vector RAG's Stage 5 query expansion
# (its EXPANSION_PROMPT), reused here for Stage 3's navigation call instead of
# embedding -- targets the terminology-mismatch failure mode from inspect_miss
# above (question phrasing vs. formal filing section titles). Routed through
# this notebook's own llm_completion (not Vector RAG's genai_client) so it
# stays cost-tracked and rate-limited like every other call here.
EXPANSION_PROMPT = """You are helping a retrieval system find the right passage in a financial \
filing (10-K/10-Q). Rewrite the question below into a short expanded search query: include likely \
synonyms, exact financial line-item names, and related terms that would appear near the answer in \
the filing. Return ONLY the expanded query text - no preamble, no explanation, no quotes.

Question: {question}"""


def expand_query(question, model) -> str:
    return llm_completion(model, EXPANSION_PROMPT.format(question=question)).strip()

EXPANDED_NAVIGATION_PATH = RESULTS_DIR / "vectorless_rag_stage_navigation_expanded.jsonl"


In [59]:



# Same shape as run_navigation_all, but expands the question first (one extra
# llm_completion call per question, stage-tagged "query_expansion" for cost
# tracking) and navigates on the expanded text instead of the raw question.
# Keeps the original question in the record for reference. Resumable, same as
# run_navigation_all -- safe to re-run after a disconnect.
def run_navigation_expanded_all(df, tree_dir, out_path, model):
    global _current_doc_name, _current_financebench_id, _current_stage
    out_path.parent.mkdir(parents=True, exist_ok=True)
    completed = _load_completed_ids(out_path)
    tree_cache = {}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            continue
        if not is_tree_built(tree_dir, row.doc_name):
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED -- {row.doc_name} not indexed yet")
            continue

        print(f"[{i}/{len(df)}] {fb_id}: expanding + navigating ...")
        _current_doc_name, _current_financebench_id = row.doc_name, fb_id
        try:
            _current_stage = "query_expansion"
            expanded_query = expand_query(row.question, model)

            if row.doc_name not in tree_cache:
                tree_cache[row.doc_name] = load_tree(tree_dir, row.doc_name)
            structure = tree_cache[row.doc_name]["structure"]

            _current_stage = "navigation_expanded"
            raw_response = navigate_tree(expanded_query, structure, model)
            node_ids = parse_navigation_response(raw_response, structure)

            record = {
                "financebench_id": fb_id,
                "doc_name": row.doc_name,
                "question": row.question,
                "expanded_query": expanded_query,
                "node_ids": node_ids,
                "raw_response": raw_response,
                "timestamp": time.time(),
            }
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
        except Exception as e:
            print(f"    FAILED: {e}")


run_navigation_expanded_all(df, TREE_DIR_FLASH, EXPANDED_NAVIGATION_PATH, MODEL)


[9/150] financebench_id_02987: expanding + navigating ...
[10/150] financebench_id_07966: expanding + navigating ...
[39/150] financebench_id_00476: expanding + navigating ...
[40/150] financebench_id_01028: expanding + navigating ...
[41/150] financebench_id_00723: expanding + navigating ...
[42/150] financebench_id_00720: expanding + navigating ...
[43/150] financebench_id_01351: expanding + navigating ...
[44/150] financebench_id_01964: expanding + navigating ...
[45/150] financebench_id_01981: expanding + navigating ...
[60/150] financebench_id_10285: expanding + navigating ...
[61/150] financebench_id_00517: expanding + navigating ...
[62/150] financebench_id_01091: expanding + navigating ...
[63/150] financebench_id_00678: expanding + navigating ...
[64/150] financebench_id_01290: expanding + navigating ...
[65/150] financebench_id_00464: expanding + navigating ...
[66/150] financebench_id_00494: expanding + navigating ...
[67/150] financebench_id_00585: expanding + navigating ..

In [60]:
# Sanity check -- read a few question -> expanded_query pairs before trusting
# the comparison below.
for r in _load_jsonl(EXPANDED_NAVIGATION_PATH)[:5]:
    print(f"Q:        {r['question']}")
    print(f"expanded: {r['expanded_query']}")
    print()


Q:        What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
expanded: 3M 2018 capital expenditures CapEx cash flow statement purchases of property plant and equipment investing activities cash used for capital expenditures FY2018 USD millions

Q:        Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer in USD billions.
expanded: 3M 2018 10-K balance sheet property plant and equipment net PP&E net fixed assets consolidated balance sheet December 31 2018 USD billions

Q:        Is 3M a capital-intensive business based on FY2022 data?
expanded: 3M capital expenditures property plant and equipment net PPE capital intensity ratio FY2022 10-K consolidated balance sheet cash flow statement capital additions spending on property plant and

In [28]:
# Same hit-check as the recall_df cell above (gold page covered by any
# selected node's page range) -- reused here so original vs. expanded
# navigation are scored identically.

def score_navigation(records):
    rows = []
    for r in records:
        fb_id, doc_name = r["financebench_id"], r["doc_name"]
        gold_row = df.loc[df.financebench_id == fb_id].iloc[0]
        gold_pages = {e["evidence_page_num"] for e in gold_row.evidence}

        if doc_name not in tree_cache:
            tree_cache[doc_name] = load_tree(TREE_DIR_FLASH, doc_name)
        node_map = create_node_mapping(tree_cache[doc_name]["structure"])

        covered_pages = set()
        for node_id in r["node_ids"]:
            node = node_map.get(node_id)
            if node is None:
                continue
            covered_pages.update(range(node["start_index"] - 1, node["end_index"]))

        rows.append({
            "financebench_id": fb_id,
            "n_nodes_selected": len(r["node_ids"]),
            "hit": bool(gold_pages & covered_pages),
        })
    return pd.DataFrame(rows)


expanded_nav_records = _load_jsonl(EXPANDED_NAVIGATION_PATH)
original_score = score_navigation(nav_records).rename(columns={"hit": "hit_original", "n_nodes_selected": "n_nodes_original","gold_pages": sorted(gold_pages)})
expanded_score = score_navigation(expanded_nav_records).rename(columns={"hit": "hit_expanded", "n_nodes_selected": "n_nodes_expanded","gold_pages": sorted(gold_pages)})

comparison = original_score.merge(expanded_score, on="financebench_id")

print(f"original hit rate: {comparison['hit_original'].mean():.3f}")
print(f"expanded hit rate: {comparison['hit_expanded'].mean():.3f}")
print(f"fixed by expansion:  {len(comparison[~comparison.hit_original & comparison.hit_expanded])}")
print(f"broken by expansion: {len(comparison[comparison.hit_original & ~comparison.hit_expanded])}")

comparison


original hit rate: 0.927
expanded hit rate: 0.940
fixed by expansion:  2
broken by expansion: 0


,financebench_id,n_nodes_original,hit_original,n_nodes_expanded,hit_expanded
0,financebench_id_03029,2,True,3,True
1,financebench_id_04672,2,True,2,True
2,financebench_id_00499,4,True,3,True
3,financebench_id_01226,2,True,2,True
4,financebench_id_01865,2,True,2,True
...,...,...,...,...,...
145,financebench_id_00735,3,True,3,True
146,financebench_id_01328,4,True,3,True
147,financebench_id_03620,2,True,4,True
148,financebench_id_04481,3,True,5,True


In [62]:

nav_by_id = {r["financebench_id"]: r for r in nav_records}
exp_by_id = {r["financebench_id"]: r for r in expanded_nav_records}

rows = []
for fb_id, exp in exp_by_id.items():
    orig = nav_by_id.get(fb_id)
    if orig is None:
        continue
    rows.append({
        "financebench_id": fb_id,
        "same_question_text": orig.get("question", "") == exp.get("question", ""),  # sanity check
        "same_nodes_picked": set(orig["node_ids"]) == set(exp["node_ids"]),
        "orig_nodes": orig["node_ids"],
        "exp_nodes": exp["node_ids"],
    })

diff_df = pd.DataFrame(rows)
print(f"identical node picks: {diff_df['same_nodes_picked'].mean():.1%}")



for r in expanded_nav_records[:5]:
    print(f"Q:        {r['question']}")
    print(f"expanded: {r['expanded_query']}")
    print(f"same:     {r['question'].strip() == r['expanded_query'].strip()}\n")


identical node picks: 23.3%
Q:        What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
expanded: 3M 2018 capital expenditures CapEx cash flow statement purchases of property plant and equipment investing activities cash used for capital expenditures FY2018 USD millions
same:     False

Q:        Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer in USD billions.
expanded: 3M 2018 10-K balance sheet property plant and equipment net PP&E net fixed assets consolidated balance sheet December 31 2018 USD billions
same:     False

Q:        Is 3M a capital-intensive business based on FY2022 data?
expanded: 3M capital expenditures property plant and equipment net PPE capital intensity ratio FY2022 10-K consolidated balance sheet cash flo

In [42]:
expanded_nav_records

[{'financebench_id': 'financebench_id_03029',
  'doc_name': '3M_2018_10K',
  'question': 'What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.',
  'expanded_query': '3M 2018 capital expenditures CapEx cash flow statement purchases of property plant and equipment investing activities cash used for capital expenditures FY2018 USD millions',
  'node_ids': ['0012', '0010', '0009'],
  'raw_response': '{\n  "thinking": "The question asks for 2018 capital expenditures (CapEx), which is a standard line item on the Statement of Cash Flows (under investing activities, typically titled \'Purchases of property, plant and equipment\'). Item 8 contains the consolidated financial statements, which include the Statement of Cash Flows. Item 7 (MD&A) frequently discusses capital expenditure trends and totals to explain liquidity and investment activities for the year.",\n  "node_ids": ["0012", 

In [28]:
diff_df

,financebench_id,same_question_text,same_nodes_picked,orig_nodes,exp_nodes
0,financebench_id_03029,False,False,"[0012, 0010]","[0012, 0010, 0009]"
1,financebench_id_04672,False,True,"[0012, 0009]","[0012, 0009]"
2,financebench_id_00499,False,False,"[0009, 0011, 0004, 0001]","[0011, 0009, 0001]"
3,financebench_id_01226,False,True,"[0009, 0011]","[0009, 0011]"
4,financebench_id_01865,False,False,"[0009, 0001]","[0009, 0011]"
...,...,...,...,...,...
143,financebench_id_00215,False,False,"[0025, 0045, 0047, 0026, 0004, 0008, 0070, 0096]","[0025, 0047, 0050, 0069, 0096, 0067, 0070, 0026]"
144,financebench_id_00566,False,False,"[0069, 0098, 0099, 0100, 0101, 0025, 0046]","[0098, 0099, 0069, 0045, 0114, 0115, 0100]"
145,financebench_id_06247,False,False,"[0038, 0039, 0045, 0020]","[0038, 0044, 0045, 0046, 0066, 0028]"
146,financebench_id_04784,False,False,"[0056, 0046, 0045]","[0046, 0040, 0056, 0048, 0047]"


In [29]:
comparison[comparison['hit_original']!=comparison['hit_expanded']].sort_values(by='financebench_id')[['financebench_id', 'hit_original', 'hit_expanded', 'n_nodes_original', 'n_nodes_expanded']]

,financebench_id,hit_original,hit_expanded,n_nodes_original,n_nodes_expanded
117,financebench_id_00080,True,False,5,6
63,financebench_id_00464,True,False,2,5
61,financebench_id_00678,False,True,5,6
62,financebench_id_01290,False,True,2,8
71,financebench_id_01346,False,True,2,4


---
## Stage 4 — Generate the answer from the selected sections' raw text

Only the raw text of the node(s) Stage 3 selected is passed to the generator
— never the whole document. Extracted directly from Stage 2's cached
`page_texts` by the node's `start_index`/`end_index` (1-indexed, inclusive).

**Resumable** — needs Stage 3 to have produced a navigation record for the
question first.

In [10]:
def fetch_node_text(node_ids, node_map, page_texts) -> list[dict]:
    sections, seen = [], set()
    for node_id in node_ids:
        node = node_map.get(node_id)
        if node is None:
            continue
        for page in range(node["start_index"], node["end_index"] + 1):
            if page in seen:
                continue
            seen.add(page)
            sections.append({"title": node["title"], "page": page, "text": page_texts[page - 1]})
    return sections

In [11]:
# GENERATION_PROMPT = """You are a financial analyst answering a question using only the sections \
# below from a company's SEC filing. Answer concisely and precisely, matching the format the question \
# expects (a number, a yes/no with brief reasoning, etc). If the sections don't contain enough \
# information to answer, say so explicitly rather than guessing.

# Question: {question}

# Sections:
# {sections}

# Answer:"""


# # Walkthrough's Stage 4, exactly: build the prompt, ask the model, return
# # the answer.
# def generate_answer(question, sections_text, model) -> str:
#     prompt = GENERATION_PROMPT.format(question=question, sections=sections_text)
#     return llm_completion(model, prompt)


GENERATION_PROMPT = """You are a financial analyst answering a question using only the sections \
below from a company's SEC filing. Each section is labeled with its title and page number.

Preserve units and currency exactly as stated in the source (do not convert between millions and \
thousands, or between currencies). If the question requires a calculation across multiple figures, \
show the calculation briefly before giving the final answer. If the sections don't contain enough \
information to answer, say so explicitly rather than guessing.

Question: {question}

Sections:
{sections}

Respond in this exact JSON format:
{{
  "reasoning": "<brief reasoning, including any calculation and which section(s) you drew from>",
  "answer": "<the final answer, matching the format the question expects>",
  "cited_pages": [<page numbers you drew from>]
}}"""


def format_sections(sections: list[dict]) -> str:
    """sections: list of {"title": str, "page": int, "text": str}"""
    return "\n\n".join(
        f"[Page {s['page']}, Section: {s['title']}]\n{s['text']}"
        for s in sections
    )


def generate_answer(question, sections: list[dict], model) -> dict:
    if not sections:
        return {"reasoning": "No sections retrieved.", "answer": "Insufficient information", "cited_pages": []}
    sections_text = format_sections(sections)
    prompt = GENERATION_PROMPT.format(question=question, sections=sections_text)
    response = llm_completion(model, prompt)
    return json.loads(response)

In [14]:
# The loop: for each question that has a navigation result but no answer
# yet, load its document's tree (cached so it's not re-read for every
# question on the same doc), fetch the section text, generate, save.
def run_generation_all(df, tree_dir, navigation_path, out_path, model):
    global _current_doc_name, _current_financebench_id, _current_stage
    out_path.parent.mkdir(parents=True, exist_ok=True)
    nav_records = {r["financebench_id"]: r for r in _load_jsonl(navigation_path)}
    completed = _load_completed_ids(out_path)
    tree_cache = {}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            continue
        if fb_id not in nav_records:
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED -- no navigation output yet")
            continue

        print(f"[{i}/{len(df)}] {fb_id}: generating ...")
        _current_doc_name, _current_financebench_id, _current_stage = row.doc_name, fb_id, "generation"
        try:
            if row.doc_name not in tree_cache:
                tree_cache[row.doc_name] = load_tree(tree_dir, row.doc_name)
            tree = tree_cache[row.doc_name]
            node_map = create_node_mapping(tree["structure"])
            sections_text = fetch_node_text(nav_records[fb_id]["node_ids"], node_map, tree["page_texts"])
            answer = generate_answer(row.question, sections_text, model)
            record = {"financebench_id": fb_id, "doc_name": row.doc_name, "model_answer": answer, "timestamp": time.time()}
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
        except Exception as e:
            print(f"    FAILED: {e}")


run_generation_all(df, TREE_DIR_FLASH, EXPANDED_NAVIGATION_PATH, GENERATION_PATH, MODEL)

---
## Stage 5 — Preview

No scoring yet (see the scope note in Stage 0) — just a look at what's been generated so far, next to the gold answer.

In [15]:
generation_records = pd.DataFrame(_load_jsonl(GENERATION_PATH))
preview = df.merge(generation_records, on=["financebench_id", "doc_name"])[
    ["financebench_id", "doc_name", "question", "answer", "model_answer"]
]
print(f"{len(preview)}/{len(df)} questions have a generated answer so far")
preview.head(10)

150/150 questions have a generated answer so far


,financebench_id,doc_name,question,answer,model_answer
0,financebench_id_03029,3M_2018_10K,What is the FY2018 capital expenditure amount ...,$1577.00,{'reasoning': 'To determine the FY2018 capital...
1,financebench_id_04672,3M_2018_10K,Assume that you are a public equities analyst....,$8.70,{'reasoning': 'The Consolidated Balance Sheet ...
2,financebench_id_00499,3M_2022_10K,Is 3M a capital-intensive business based on FY...,"No, the company is managing its CAPEX and Fixe...",{'reasoning': 'A capital-intensive business is...
3,financebench_id_01226,3M_2022_10K,What drove operating margin change as of FY202...,Operating Margin for 3M in FY2022 has decrease...,{'reasoning': 'According to the Operating Inco...
4,financebench_id_01865,3M_2022_10K,"If we exclude the impact of M&A, which segment...",The consumer segment shrunk by 0.9% organically.,{'reasoning': 'To identify which segment dragg...
5,financebench_id_00807,3M_2023Q2_10Q,Does 3M have a reasonably healthy liquidity pr...,No. The quick ratio for 3M was 0.96 by Jun'23 ...,{'reasoning': 'The quick ratio is calculated a...
6,financebench_id_00941,3M_2023Q2_10Q,Which debt securities are registered to trade ...,Following debt securities registered under 3M'...,{'reasoning': 'According to the 'Cover Page' s...
7,financebench_id_01858,3M_2023Q2_10Q,Does 3M maintain a stable trend of dividend di...,"Yes, not only they distribute the dividends on...","{'reasoning': 'According to page 73, 3M has pa..."
8,financebench_id_02987,ACTIVISIONBLIZZARD_2019_10K,What is the FY2019 fixed asset turnover ratio ...,24.26,{'reasoning': 'The fixed asset turnover ratio ...
9,financebench_id_07966,ACTIVISIONBLIZZARD_2019_10K,What is the FY2017 - FY2019 3 year average of ...,1.9%,{'reasoning': 'To calculate the 3-year average...


---
## Stage 6 — Score each answer

Deterministic numeric matching first (handles unit differences and small
rounding); falls through to an LLM judge (`openai/gpt-oss-120b` via Groq's
free tier — a different model family from Gemini, which matters for avoiding
self-enhancement bias) only when a question isn't numeric. Reuses
`evaluation/answer_scorer.py`, the same scorer `vector_rag_pipeline.ipynb`
uses, so both pipelines are graded identically. Spot-check 10-15% of judge
outputs by hand before trusting them for the real experiment.

**Resumable** — needs Stage 4 to have produced a generated answer for the
question first.

In [ ]:
# from groq import Groq
# from evaluation.answer_scorer import score_answer

# judge_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
# JUDGE_MODEL = "openai/gpt-oss-120b"


# # For each question with a generated answer but no score yet: run the
# # deterministic matcher, fall back to the LLM judge if that's inconclusive,
# # save the label. Judge token usage AND latency (when the judge was actually
# # called -- the deterministic path makes no API call, so timing it would just
# # log ~0s noise) gets logged to cost_tracker same as everything else.
# def run_scoring_all(df, generation_path, out_path, judge_client, judge_model):
#     global _current_doc_name, _current_financebench_id, _current_stage
#     out_path.parent.mkdir(parents=True, exist_ok=True)
#     gen_records = {r["financebench_id"]: r for r in _load_jsonl(generation_path)}
#     completed = _load_completed_ids(out_path)

#     for i, row in enumerate(df.itertuples(), start=1):
#         fb_id = row.financebench_id
#         if fb_id in completed:
#             continue
#         if fb_id not in gen_records:
#             print(f"[{i}/{len(df)}] {fb_id}: SKIPPED -- no generated answer yet")
#             continue

#         print(f"[{i}/{len(df)}] {fb_id}: scoring ...")
#         _current_doc_name, _current_financebench_id, _current_stage = row.doc_name, fb_id, "judge"
#         try:
#             model_answer = gen_records[fb_id]["model_answer"]
#             t0 = time.time()
#             result, usage = score_answer(
#                 row.question, row.answer, model_answer,
#                 judge_client=judge_client, judge_model=judge_model,
#             )
#             judge_latency = time.time() - t0
#             if usage:
#                 cost_tracker.log(
#                     pipeline="vectorless_rag", stage="judge", model=judge_model,
#                     input_tokens=usage["input_tokens"], output_tokens=usage["output_tokens"],
#                     doc_name=row.doc_name, financebench_id=fb_id,
#                     latency_sec=judge_latency,
#                 )
#             record = {
#                 "financebench_id": fb_id, "doc_name": row.doc_name,
#                 "label": result.label, "method": result.method, "reasoning": result.reasoning,
#             }
#             with out_path.open("a") as f:
#                 f.write(json.dumps(record) + "\n")
#         except Exception as e:
#             print(f"    FAILED: {e}")


# run_scoring_all(df, GENERATION_PATH, SCORING_PATH, judge_client, JUDGE_MODEL)


[9/150] financebench_id_02987: scoring ...
[10/150] financebench_id_07966: scoring ...
[14/150] financebench_id_00438: scoring ...
[15/150] financebench_id_00591: scoring ...
[16/150] financebench_id_01319: scoring ...
[17/150] financebench_id_00540: scoring ...
[18/150] financebench_id_10420: scoring ...
[48/150] financebench_id_00070: scoring ...
[61/150] financebench_id_00517: scoring ...
[62/150] financebench_id_01091: scoring ...
[63/150] financebench_id_00678: scoring ...
[64/150] financebench_id_01290: scoring ...
[65/150] financebench_id_00464: scoring ...
[66/150] financebench_id_00494: scoring ...
[67/150] financebench_id_00585: scoring ...
[76/150] financebench_id_05915: scoring ...
[95/150] financebench_id_00299: scoring ...
[96/150] financebench_id_02119: scoring ...
[97/150] financebench_id_00206: scoring ...
[120/150] financebench_id_04980: scoring ...
[121/150] financebench_id_01009: scoring ...
[122/150] financebench_id_00735: scoring ...
[123/150] financebench_id_0132

In [16]:
from groq import Groq
from evaluation.answer_scorer import score_deterministic, score_with_judge

judge_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
JUDGE_MODEL = "openai/gpt-oss-120b"

def extract_answer_fields(model_answer):
    """Generator now returns {"reasoning", "answer", "cited_pages"}; older records were plain strings."""
    if isinstance(model_answer, dict):
        return str(model_answer.get("answer", "")), model_answer.get("reasoning"), model_answer.get("cited_pages")
    return str(model_answer), None, None



In [ ]:
def run_scoring_all(df, generation_path, out_path, judge_client, judge_model):
      global _current_doc_name, _current_financebench_id, _current_stage
      out_path.parent.mkdir(parents=True, exist_ok=True)
      gen_records = {r["financebench_id"]: r for r in _load_jsonl(generation_path)}
      completed = _load_completed_ids(out_path)

      for i, row in enumerate(df.itertuples(), start=1):
          fb_id = row.financebench_id
          if fb_id in completed:
              continue
          if fb_id not in gen_records:
              print(f"[{i}/{len(df)}] {fb_id}: SKIPPED -- no generated answer yet")
              continue

          print(f"[{i}/{len(df)}] {fb_id}: scoring ...")
          _current_doc_name, _current_financebench_id, _current_stage = row.doc_name, fb_id, "judge"
          try:
              answer_text, reasoning, cited_pages = extract_answer_fields(gen_records[fb_id]["model_answer"])
              is_metric = row.question_type == "metrics-generated"

              det_label, det_gold, det_matched = "NA", None, None
              if is_metric:
                  det = score_deterministic(row.question, row.answer, answer_text)
                  if det is None:
                      det_label = "Inconclusive"   # no extractable number in gold and/or answer
                  else:
                      det_label, det_gold, det_matched = det.label, det.gold_value, det.matched_value

              judge_label, judge_reasoning = None, None
              if det_label != "Correct":
                  t0 = time.time()
                  result, usage = score_with_judge(row.question, row.answer, answer_text, judge_client, judge_model)
                  judge_latency = time.time() - t0
                  judge_label, judge_reasoning = result.label, result.reasoning
                  cost_tracker.log(
                      pipeline="vectorless_rag", stage="judge", model=judge_model,
                      input_tokens=usage["input_tokens"], output_tokens=usage["output_tokens"],
                      doc_name=row.doc_name, financebench_id=fb_id, latency_sec=judge_latency,
                  )

              record = {
                  "financebench_id": fb_id, "doc_name": row.doc_name,
                  "question_type": row.question_type, "question": row.question,
                  "gold_answer": row.answer, "model_answer": answer_text,
                  "model_reasoning": reasoning, "cited_pages": cited_pages,
                  "deterministic_label": det_label,
                  "deterministic_gold_value": det_gold, "deterministic_matched_value": det_matched,
                  "judge_label": judge_label, "judge_reasoning": judge_reasoning,
                  # final verdict, so summarize_results (Stage 9) keeps working
                  "label": "Correct" if det_label == "Correct" else judge_label,
                  "method": "deterministic" if det_label == "Correct" else "llm_judge",
                  "timestamp": time.time(),
              }
              with out_path.open("a") as f:
                  f.write(json.dumps(record) + "\n")
          except Exception as e:
              print(f"    FAILED: {e}")


(df, GENERATION_PATH, SCORING_PATH, judge_client, JUDGE_MODEL)



[89/150] financebench_id_00651: scoring ...
[143/150] financebench_id_00859: scoring ...


---
## Stage 8 — Dedicated latency timing pass

A small, **strictly sequential** pass over a fixed sample, run fully fresh
each time (navigate → fetch section text → generate, timed end-to-end and
per sub-stage) — matching the project brief's "run each query multiple
times, report the median." Tree-building (Stage 2) is excluded: that's
one-time preprocessing per document, not something a live query pays for, so
a question can only be sampled here if its document is already indexed.
Scoring is excluded too — grading isn't part of response time.

**Costs real API calls beyond Stages 2-4** — every (question, repeat) pair
here is a fresh navigate+generate call, not reused from earlier stages.
Default is a conservative **5 questions × 3 repeats = 15 timed passes** —
raise `LATENCY_SAMPLE_SIZE`/`LATENCY_REPEATS` only once you're ready to spend
more on this specific stage. It's resumable in the sense that a
(question, repeat) pair already timed is skipped, so re-running this cell
without changing the sample doesn't cost anything more.

In [21]:
import time
import statistics

LATENCY_SAMPLE_SIZE = 10  # matches vector RAG's Stage 8 sample size, so the two are directly comparable
LATENCY_REPEATS = 3       # per project brief: run each query multiple times, report the median


# One fully-fresh, uncached pass: navigate -> fetch selected section text ->
# generate. Returns (timings_sec dict, model_answer).
def time_single_pass(row, tree, node_map, model):
    global _current_doc_name, _current_financebench_id, _current_stage
    _current_doc_name, _current_financebench_id = row.doc_name, row.financebench_id
    timings = {}
    t_total0 = time.time()

    _current_stage = "latency_navigation"
    t0 = time.time()
    node_ids = navigate_tree(row.question, tree["structure"], model)
    timings["navigate"] = time.time() - t0

    t0 = time.time()
    sections_text = fetch_node_text(node_ids, node_map, tree["page_texts"])
    timings["fetch_section_text"] = time.time() - t0

    _current_stage = "latency_generation"
    t0 = time.time()
    answer = generate_answer(row.question, sections_text, model)
    timings["generate"] = time.time() - t0

    timings["total"] = time.time() - t_total0
    return timings, answer

In [22]:
# Samples a fixed set of already-indexed questions and times each one
# `repeats` times, skipping (question, repeat) pairs already timed.
def run_latency_pass(df, tree_dir, model, out_path, sample_size=LATENCY_SAMPLE_SIZE, repeats=LATENCY_REPEATS, seed=42):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    indexed = df[df.doc_name.apply(lambda d: is_tree_built(tree_dir, d))]
    if len(indexed) < len(df):
        print(f"Note: {len(df) - len(indexed)} questions excluded -- their document isn't indexed yet")
    sample = indexed.sample(n=min(sample_size, len(indexed)), random_state=seed)

    completed_pairs = {(r["financebench_id"], r["repeat"]) for r in _load_jsonl(out_path)}
    total_runs = len(sample) * repeats
    print(f"Timing {len(sample)} questions x {repeats} repeats = {total_runs} passes "
          f"({len(completed_pairs)} already done)")

    tree_cache = {}
    for row in sample.itertuples():
        if row.doc_name not in tree_cache:
            tree = load_tree(tree_dir, row.doc_name)
            tree_cache[row.doc_name] = (tree, create_node_mapping(tree["structure"]))
        tree, node_map = tree_cache[row.doc_name]

        for repeat in range(1, repeats + 1):
            if (row.financebench_id, repeat) in completed_pairs:
                continue
            print(f"  {row.financebench_id} repeat {repeat}/{repeats} ...")
            try:
                timings, answer = time_single_pass(row, tree, node_map, model)
                record = {
                    "financebench_id": row.financebench_id, "doc_name": row.doc_name,
                    "repeat": repeat, "timings_sec": timings, "model_answer": answer,
                }
                with out_path.open("a") as f:
                    f.write(json.dumps(record) + "\n")
            except Exception as e:
                print(f"    FAILED: {e}")


run_latency_pass(df, TREE_DIR_FLASH, MODEL, LATENCY_PATH)

Timing 5 questions x 3 repeats = 15 passes (15 already done)
  financebench_id_00005 repeat 1/3 ...
    FAILED: name 'navigate_tree' is not defined
  financebench_id_00005 repeat 2/3 ...
    FAILED: name 'navigate_tree' is not defined
  financebench_id_00005 repeat 3/3 ...
    FAILED: name 'navigate_tree' is not defined
  financebench_id_06655 repeat 1/3 ...
    FAILED: name 'navigate_tree' is not defined
  financebench_id_06655 repeat 2/3 ...
    FAILED: name 'navigate_tree' is not defined
  financebench_id_06655 repeat 3/3 ...
    FAILED: name 'navigate_tree' is not defined
  financebench_id_00080 repeat 1/3 ...
    FAILED: name 'navigate_tree' is not defined
  financebench_id_00080 repeat 2/3 ...
    FAILED: name 'navigate_tree' is not defined
  financebench_id_00080 repeat 3/3 ...
    FAILED: name 'navigate_tree' is not defined
  financebench_id_01244 repeat 1/3 ...
    FAILED: name 'navigate_tree' is not defined
  financebench_id_01244 repeat 2/3 ...
    FAILED: name 'navigate_tre

In [40]:
# Median-of-N per question (per the project brief), then median across
# questions for the headline number -- robust to one slow/retried pass
# dragging the number the way a mean would.
def summarize_latency(out_path):
    records = _load_jsonl(out_path)
    if not records:
        print(f"No latency data yet in {out_path}")
        return {}

    by_question = {}
    for r in records:
        by_question.setdefault(r["financebench_id"], []).append(r["timings_sec"]["total"])
    per_question_median = {fb_id: statistics.median(vals) for fb_id, vals in by_question.items()}
    overall_median = statistics.median(per_question_median.values())

    stage_names = [k for k in records[0]["timings_sec"] if k != "total"]
    stage_medians = {
        stage: statistics.median(r["timings_sec"][stage] for r in records) for stage in stage_names
    }
    print(f"Median end-to-end latency ({len(per_question_median)} questions): {overall_median:.2f}s")
    for stage, med in stage_medians.items():
        print(f"  median {stage}: {med:.2f}s")
    return {"overall_median_sec": overall_median, "stage_medians_sec": stage_medians}


summarize_latency(LATENCY_PATH)

Median end-to-end latency (5 questions): 1.10s
  median navigate: 0.46s
  median fetch_section_text: 0.00s
  median generate: 0.68s


{'overall_median_sec': 1.0969908237457275,
 'stage_medians_sec': {'navigate': 0.45878148078918457,
  'fetch_section_text': 9.417533874511719e-05,
  'generate': 0.6802136898040771}}

---
## Stage 9 — Summarize

Answer-quality breakdown (Stage 6), navigation hit rate (did the selected nodes cover a gold page), and
token/cost totals by stage (Stage 2's indexing calls onward) — computed over
whatever's done so far in each stage's output file, not requiring all 150
questions to be finished.

In [29]:
from collections import Counter


def summarize_results(scoring_path, cost_path):
    scoring = _load_jsonl(scoring_path)
    if scoring:
        n = len(scoring)
        label_counts = Counter(r["label"] for r in scoring)
        print(f"Answer quality ({n} scored questions):")
        for label, count in label_counts.items():
            print(f"  {label}: {count} ({100 * count / n:.1f}%)")
    else:
        print(f"No scored results yet in {scoring_path}")

    print()
    # No ranking here (PageIndex picks a set of nodes), so retrieval quality is
    # a hit rate: did any selected node's page range cover a gold page?
    for name, nav_path in [("original", NAVIGATION_PATH), ("expanded", EXPANDED_NAVIGATION_PATH)]:
        nav = _load_jsonl(nav_path)
        if nav:
            hit = score_navigation(nav)["hit"]
            print(f"Navigation hit rate, {name} ({len(hit)} questions): {hit.mean():.3f}")
    print()
    costs = _load_jsonl(cost_path)
    if costs:
        tokens_by_stage = Counter()
        cost_by_stage = Counter()
        for r in costs:
            tokens_by_stage[r["stage"]] += r["input_tokens"] + r["output_tokens"]
            if r["cost_usd"] is not None:
                cost_by_stage[r["stage"]] += r["cost_usd"]
        print(f"Tokens/cost by stage ({len(costs)} logged calls):")
        for stage, tokens in tokens_by_stage.items():
            cost_str = f"${cost_by_stage[stage]:.4f}" if stage in cost_by_stage else "no price set"
            print(f"  {stage}: {tokens:,} tokens, {cost_str}")
        print(f"  TOTAL: ${sum(cost_by_stage.values()):.4f}")
    else:
        print(f"No cost data yet in {cost_path}")


summarize_results(SCORING_PATH, COSTS_PATH)

Answer quality (150 scored questions):
  Correct: 119 (79.3%)
  Incorrect: 28 (18.7%)
  Failure to Answer: 3 (2.0%)

Navigation hit rate, original (150 questions): 0.927
Navigation hit rate, expanded (150 questions): 0.940

Tokens/cost by stage (13572 logged calls):
  indexing_flash: 23,131,096 tokens, $9.3569
  unknown: 6,397,287 tokens, $1.6918
  navigation: 2,030,420 tokens, $0.5437
  query_expansion: 26,600 tokens, $0.0160
  navigation_expanded: 2,012,614 tokens, $0.5453
  generation: 13,449,034 tokens, $3.4651
  judge: 50,475 tokens, $0.0000
  TOTAL: $15.6187


---
## Clean cost breakdown (deduplicated)

`summarize_results` above sums the raw cost log as-is -- but `COSTS_PATH` is
append-only with no "already logged this question" check, so re-running a
cell (retries, a `df.head(5)` smoke test after the real run, re-opening the
notebook) appends *more* records for the same question without replacing
anything. A real check of this project's cost files found stages with up to
14 duplicate records for a single question -- not a rare edge case.

This cell uses `evaluation/cost_tracker.py`'s `clean_stage_costs` to match
each question's cost record to the one that actually produced its *saved*
answer (nearest-timestamp join against that stage's own resumable output
file), dropping everything else as noise, then `recompute_cost_usd` to price
every record fresh against the *current* pricing table -- not whatever
`cost_usd` was frozen in at log time, which matters if a price was added,
fixed, or wrong when the record was first written.

**Only counts the navigation path actually used for generation** (expanded,
per `GENERATION_PATH`'s input) -- the un-expanded `NAVIGATION_PATH` run is
real exploratory work but isn't part of this pipeline's final per-query
cost, so it's reported separately, not summed in.

**Excludes `latency_navigation` / `latency_generation`** -- those are calls
made by the dedicated timing pass below, re-answering a 10-question sample
for measurement, not new answers to the 150. Real money, but not part of
"cost to answer 150 questions" -- shown separately.

In [ ]:
from evaluation.cost_tracker import clean_stage_costs, clean_doc_level_costs, recompute_cost_usd

costs = _load_jsonl(COSTS_PATH)
gen_out = {r["financebench_id"]: r for r in _load_jsonl(GENERATION_PATH)}
nav_exp_out = {r["financebench_id"]: r for r in _load_jsonl(EXPANDED_NAVIGATION_PATH)}
scoring_all = _load_jsonl(SCORING_PATH)
judge_out = {r["financebench_id"]: r for r in scoring_all if r.get("judge_label") is not None}

n = len(gen_out)
print(f"Cleaning against {n} generated answers\n")

clean_records = []
for stage_name, out in [
    ("navigation_expanded", nav_exp_out),   # the navigation actually used for generation
    ("query_expansion", nav_exp_out),       # happens in the same loop iteration, right before it
    ("generation", gen_out),
    ("judge", judge_out),
]:
    kept, report = clean_stage_costs(costs, stage_name, out)
    print(f"{stage_name:<20} kept {report['n_kept']}/{report['n_input']} "
          f"(dropped {report['n_dropped_as_noise']} as noise)"
          + (f"  MISSING for: {report['missing_cost']}" if report["missing_cost"] else "")
          + (f"  late_only: {report['late_only']}" if report["late_only"] else ""))
    clean_records += kept

for stage_name in ["indexing_flash", "indexing_fallback"]:
    kept, report = clean_doc_level_costs(costs, stage_name)
    print(f"{stage_name:<20} kept {report['n_kept']}/{report['n_input']} docs "
          f"(dropped {report['n_dropped_as_noise']} as noise)"
          + (f"  ambiguous (rebuilt more than once): {report['ambiguous_docs']}" if report["ambiguous_docs"] else ""))
    clean_records += kept

clean_records = recompute_cost_usd(clean_records)

print("\nClean tokens/cost by stage:")
by_stage = {}
for r in clean_records:
    d = by_stage.setdefault(r["stage"], {"tok": 0, "cost": 0.0, "unpriced": 0, "n": 0})
    d["tok"] += r["input_tokens"] + r["output_tokens"]; d["n"] += 1
    if r["cost_usd"] is not None:
        d["cost"] += r["cost_usd"]
    else:
        d["unpriced"] += 1
for stage, d in by_stage.items():
    unpriced_n = d["unpriced"]
    note = f"  ({unpriced_n} unpriced)" if unpriced_n else ""
    print(f"  {stage:<20} n={d['n']:<4} tokens={d['tok']:>10,}  ${d['cost']:.4f}{note}")

per_query_stages = {"navigation_expanded", "query_expansion", "generation", "judge"}
indexing_stages = {"indexing_flash", "indexing_fallback"}
per_query_total = sum(d["cost"] for s, d in by_stage.items() if s in per_query_stages)
indexing_total = sum(d["cost"] for s, d in by_stage.items() if s in indexing_stages)
n_docs_indexed = sum(d["n"] for s, d in by_stage.items() if s in indexing_stages)
print(f"\nPer-query cost : ${per_query_total:.4f} total  (${per_query_total / n:.5f} / question, over {n} questions)")
print(f"Indexing cost  : ${indexing_total:.4f} total, one-time (over {n_docs_indexed} documents)")

# The un-expanded navigation run, for reference only -- not part of the totals above
nav_out = {r["financebench_id"]: r for r in _load_jsonl(NAVIGATION_PATH)}
if nav_out:
    kept_nav, _ = clean_stage_costs(costs, "navigation", nav_out)
    kept_nav = recompute_cost_usd(kept_nav)
    tok = sum(r["input_tokens"] + r["output_tokens"] for r in kept_nav)
    cost = sum(r["cost_usd"] or 0.0 for r in kept_nav)
    print(f"\n(Un-expanded navigation, NOT part of the pipeline's cost -- exploratory only: "
          f"{len(kept_nav)} calls, {tok:,} tokens, ${cost:.4f})")

In [30]:
# Per-question breakdown: navigation hit/miss vs answer correct/wrong, with the
# question type and both scorer verdicts (deterministic for metric questions, LLM judge).
# Show full text in every cell and don't collapse rows (pandas cuts at 50 chars / 60 rows by default)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

nav = score_navigation(_load_jsonl(EXPANDED_NAVIGATION_PATH))[["financebench_id", "hit"]]
scores = pd.DataFrame(_load_jsonl(SCORING_PATH))

full = scores.merge(nav, on="financebench_id", how="left")
full["nav"] = full["hit"].map({True: "Nav hit", False: "Nav missed"})
full["answer"] = full["label"].map(lambda l: "Correct" if l == "Correct" else "Wrong")   # Incorrect + Failure to Answer

def bucket(r):
    if r.nav == "Nav missed":
        return "1. Nav missed"
    return "2. Nav hit, answer wrong" if r.answer == "Wrong" else "3. Nav hit, answer correct"
full["bucket"] = full.apply(bucket, axis=1)

# Judge column is empty when deterministic already said Correct (judge skipped)
full["judge_label"] = full["judge_label"].fillna("not run")

# Summary counts
print(pd.crosstab(full["nav"], full["answer"], margins=True))
print()
print(full["bucket"].value_counts().sort_index().to_string())

# All 150 questions, problem groups first
cols = ["bucket", "financebench_id", "doc_name",
        "question_type",            # metrics-generated / domain-relevant / novel-generated
        "question", "gold_answer", "model_answer",
        "deterministic_label",      # Correct/Incorrect/Inconclusive for metric questions, NA otherwise
        "deterministic_gold_value", "deterministic_matched_value",
        "judge_label", "judge_reasoning",
        "label"]                    # final verdict
full.sort_values(["bucket", "question_type", "doc_name"])[cols].reset_index(drop=True)

answer      Correct  Wrong  All
nav                            
Nav hit         114     27  141
Nav missed        5      4    9
All             119     31  150

bucket
1. Nav missed                   9
2. Nav hit, answer wrong       27
3. Nav hit, answer correct    114


,bucket,financebench_id,doc_name,question_type,question,gold_answer,model_answer,deterministic_label,deterministic_gold_value,deterministic_matched_value,judge_label,judge_reasoning,label
0,1. Nav missed,financebench_id_00790,CVSHEALTH_2022_10K,domain-relevant,Is CVS Health a capital-intensive business bas...,"Yes, CVS Health requires an extensive asset ba...","Based on the FY2022 data, CVS Health does not ...",NA,NaN,NaN,Incorrect,The model answer claims CVS Health is not capi...,Incorrect
1,1. Nav missed,financebench_id_01244,CVSHEALTH_2022_10K,domain-relevant,Has CVS Health paid dividends to common shareh...,"Yes, CVS paid a $ 0.55 dividend per share ever...","Yes, CVS Health paid dividends to common share...",NA,NaN,NaN,Correct,The model confirms CVS paid a $0.55 per share ...,Correct
2,1. Nav missed,financebench_id_00206,JPMORGAN_2022_10K,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...",Gross margins are not a relevant metric for a ...,NA,NaN,NaN,Correct,The model correctly states that gross margin i...,Correct
3,1. Nav missed,financebench_id_04784,WALMART_2019_10K,metrics-generated,Based on the information provided primarily in...,0.2%,0.2%,Correct,0.2,0.2,not run,None,Correct
4,1. Nav missed,financebench_id_00494,BOEING_2022_10K,novel-generated,What production rate changes is Boeing forecas...,Boeing forecasts an increase in the production...,"For FY2023, Boeing's forecast for production r...",NA,NaN,NaN,Incorrect,The model says the 777X production remains pau...,Incorrect
...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,"3. Nav hit, answer correct",financebench_id_00702,PFIZER_2021_10K,novel-generated,Were there any potential events that are not i...,"Yes, the gain on completion of Consumer Health...","Yes, in 2019 Pfizer recognized a gain of $8.10...",NA,NaN,NaN,Correct,The model correctly identifies the Consumer He...,Correct
146,"3. Nav hit, answer correct",financebench_id_02416,PFIZER_2021_10K,novel-generated,What are three main companies acquired by Pfiz...,"Trillium, Array, and Therachon",The three main companies acquired by Pfizer me...,NA,NaN,NaN,Correct,The model's answer exactly matches the gold an...,Correct
147,"3. Nav hit, answer correct",financebench_id_00724,Pfizer_2023Q2_10Q,novel-generated,"For Pfizer, which geographic region had the bi...",Developed Rest of the World,Developed Rest of World,NA,NaN,NaN,Correct,The model's answer exactly matches the gold an...,Correct
148,"3. Nav hit, answer correct",financebench_id_00601,ULTABEAUTY_2023Q4_EARNINGS,novel-generated,What drove the reduction in SG&A expense as a ...,Lower marketing expenses and leverage of incen...,The reduction in SG&A expense as a percent of ...,NA,NaN,NaN,Correct,The model includes the key drivers from the go...,Correct


In [34]:
full.loc[full["bucket"] == "2. Nav hit, answer wrong",['gold_answer', 'model_answer']].reset_index(drop=True)

,gold_answer,model_answer
0,"No, the company is managing its CAPEX and Fixe...","Yes, 3M is a capital-intensive business. This ..."
1,No. The quick ratio for 3M was 0.96 by Jun'23 ...,Based on the quick ratio of approximately 0.96...
2,"Yes, the FCF conversion (using net income as t...","Yes, Adobe has an improving Free cash flow as ..."
3,AES has converted inventory 9.5 times in FY 2022.,AES Corporation's inventory turnover ratio for...
4,$0.40,0.389
5,"Yes, there is decline in number stores by 1.32...","Yes, there was a change. The total number of B..."
6,The entertainment segment experienced the high...,Computing and Mobile Phones
7,Yes. Boeing has an improving gross margin prof...,Gross margin is not a useful metric for Boeing...
8,Boeing's primary customers as of FY2022 are a ...,"As of FY2022, Boeing’s primary customers inclu..."
9,"Effective tax rate in FY2022 was 0.62%, compar...","In FY2022, Boeing's effective tax rate was app..."
